# BrainFormer-PD v2.1 — End-to-End Colab Notebook

**Architecture:** Stochastic Personalised Braak Graph Learning (S-PBGL) + LTP + IMPE + BSMTA + EDP/UG-HEM + NHH + GBA-BCSD

**Targets:** UPDRS-III @ 36mo (R²), NSD-ISS staging (Weighted F1), DaTscan SBR (Pearson r), time-to-milestone (C-index), uncertainty calibration (ECE).

All data/checkpoints persisted under `/content/drive/MyDrive/BRAAK_PD/` so Colab disconnects are non-destructive. Every heavy block has a resume cell.

## Block Index
- **00** Session Setup
- **01** Data Loading + Feature Engineering (A–I)
- **02** Canonical Braak Graph
- **03** Model Definition (with S-PBGL replacing PSBGL)
- **04** Datasets and DataLoaders
- **05** GBA-BCSD Contrastive Pre-training (Supervised Contrastive)
- **06** Full Multi-Task Training (KL anneal + UG-HEM)
- **07** Primary Evaluation (PD primary + prodromal secondary)
- **08** Uncertainty Calibration & Clinical Monitoring Signal
- **09** Ablation Study (B0..B5 incl. S-PBGL vs PSBGL)
- **10** Statistical Tests
- **11** GBA-Stratified Analysis
- **12** SAA-Stratified Analysis (new)
- **13** Graph Topology Phenotyping (new)
- **14** Two-Source Uncertainty Analysis (new)
- **15** Subtype Clustering (GBA-BCSD latent space)
- **16** Figures (12 original + 4 new)
- **17** Final Summary

## Block 00 — Session Setup

Mounts Drive, defines paths, fixes seeds, imports.

In [ ]:
# ==== Block 00 — installs (run once per session) ====
!pip -q install torch==2.3.0 torchvision --upgrade 2>/dev/null | tail -n 2
!pip -q install xgboost scikit-learn lifelines tqdm matplotlib seaborn pandas numpy scipy 2>/dev/null | tail -n 2
print('Installs done.')

In [ ]:
# ==== Block 00 — mount drive, paths, seeds, imports ====
import os, sys, json, pickle, random, warnings, math, time
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# --- Reproducibility ---
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

# --- Drive mount (no-op outside Colab) ---
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = '/content/drive/MyDrive/BRAAK_PD'
except Exception:
    DRIVE = str(Path.home() / 'BRAAK_PD')

ROOT       = Path(DRIVE)
DATA_RAW   = ROOT / 'data_raw'       # place PPMI CSVs here
DATA_PROC  = ROOT / 'data_proc'
SPLITS     = ROOT / 'splits'
CHECKPOINTS= ROOT / 'checkpoints'
RESULTS    = ROOT / 'results'
FIGURES    = ROOT / 'figures'
for p in [DATA_RAW, DATA_PROC, SPLITS, CHECKPOINTS, RESULTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Root:   {ROOT}')

# --- Global hyperparameters ---
CFG = dict(
    # Shapes
    d_model      = 128,
    d_visit      = 5,               # UPDRS1, UPDRS2, UPDRS3, MoCA, SBR_mean per visit
    T_MAX        = 4,               # BL, V02, V04, V06
    n_nodes      = 10,              # Braak regions
    d_node       = 8,
    n_heads      = 8,
    n_tf_layers  = 2,               # reduced from 4 (Issue 9)
    # Target scaling
    UPDRS_MAX    = 132.0,
    # Training
    pretrain_epochs = 30,
    train_epochs    = 150,
    batch_size      = 64,
    lr              = 3e-4,
    weight_decay    = 1e-5,
    grad_clip       = 1.0,
    # Loss weights
    w_updrs=1.0, w_stage=0.5, w_datscan=0.3, w_graph=0.01, w_hazard=0.1,
    kl_anneal_epochs = 30,
    kl_max           = 1e-4,
    ughem_warmup     = 20,
    ughem_gamma      = 0.5,
    # S-PBGL
    graph_scale_init = 0.1,
    kappa_init       = 0.5,
    logvar_clip      = (-4.0, 2.0),
    braak_eps        = 0.05,
)
with open(RESULTS/'config.json', 'w') as f:
    json.dump(CFG, f, indent=2)
print('Config saved.')

## Block 01 — Data Loading + Feature Engineering

Sub-blocks A through I. Each sub-block resumes from its own checkpoint in `DATA_PROC/`. If raw PPMI CSVs are missing, Block 01-A generates a deterministic synthetic PPMI-like dataset so the rest of the pipeline remains runnable; swap in real CSVs to get real results.

In [ ]:
# ==== Block 01-A — load (or synthesize) master dataframe ====
BLOCK_01A = DATA_PROC / 'master_df.parquet'
SPLIT_TR  = SPLITS / 'train_patnos.npy'
SPLIT_VA  = SPLITS / 'val_patnos.npy'
SPLIT_TE  = SPLITS / 'test_patnos.npy'

if BLOCK_01A.exists() and SPLIT_TR.exists():
    master_df = pd.read_parquet(BLOCK_01A)
    train_patnos = np.load(SPLIT_TR); val_patnos = np.load(SPLIT_VA); test_patnos = np.load(SPLIT_TE)
    print(f'Loaded master_df: {master_df.shape}, splits {len(train_patnos)}/{len(val_patnos)}/{len(test_patnos)}')
    SKIP_01A = True
else:
    SKIP_01A = False

if not SKIP_01A:
    # Try to load real PPMI
    real_candidate = DATA_RAW / 'PPMI_master.csv'
    if real_candidate.exists():
        master_df = pd.read_csv(real_candidate)
        print(f'Real PPMI loaded: {master_df.shape}')
    else:
        # ---- Synthetic PPMI-like cohort so notebook always runs ----
        print('Real PPMI not found — synthesising a deterministic 8,453-subject PPMI-like cohort.')
        rng = np.random.default_rng(SEED)
        N = 8453
        cohort = rng.choice(['PD','HC','Prodromal','SWEDD'], size=N, p=[0.25, 0.15, 0.55, 0.05])
        GBA  = rng.binomial(1, 0.042, size=N)
        SAA  = rng.binomial(1, 0.271, size=N)
        age  = rng.normal(63, 9, size=N).clip(35, 90)
        sex  = rng.binomial(1, 0.6, size=N)
        edu  = rng.normal(15, 3, size=N).clip(6, 22)
        # Latent severity — cohort and GBA driven
        sev0 = np.where(cohort=='PD', rng.normal(22, 8, N),
               np.where(cohort=='SWEDD', rng.normal(10,5,N),
               np.where(cohort=='Prodromal', rng.normal(4,3,N), rng.normal(1,1,N))))
        vel  = rng.normal(0, 2, N) + 2.5*GBA + 0.8*SAA + 0.02*(age-63)
        acc  = rng.normal(0, 0.6, N)
        def traj(v0, v1, v2, v3):
            return np.stack([v0, v0+v1, v0+2*v1+v2, v0+3*v1+3*v2], axis=1)
        updrs3 = traj(sev0, vel, acc*0.3, 0)
        updrs3 = (updrs3 + rng.normal(0, 3, updrs3.shape)).clip(0, 132)
        updrs1 = (0.3*updrs3 + rng.normal(0,2,updrs3.shape)).clip(0,52)
        updrs2 = (0.35*updrs3+ rng.normal(0,2,updrs3.shape)).clip(0,52)
        moca   = (28 - 0.15*updrs3 + rng.normal(0,1.5,updrs3.shape)).clip(0,30)
        sbr    = (2.5 - 0.03*updrs3 + rng.normal(0,0.3,updrs3.shape)).clip(0.3, 4.0)
        # Informative missingness — Prodromal: more CSF; PD: more DaTscan; random dropout
        visit_mask_updrs3 = rng.binomial(1, 0.90, updrs3.shape).astype(bool)
        visit_mask_updrs3[:,0] = True  # BL always observed
        # Imaging observed at BL and V04 mostly
        visit_mask_sbr = np.zeros(sbr.shape, dtype=bool)
        visit_mask_sbr[:,0] = rng.binomial(1, 0.75, N).astype(bool)
        visit_mask_sbr[:,2] = rng.binomial(1, 0.55, N).astype(bool)
        def apply_mask(arr, mask):
            out = arr.astype(float).copy(); out[~mask] = np.nan; return out
        updrs3 = apply_mask(updrs3, visit_mask_updrs3)
        updrs1 = apply_mask(updrs1, visit_mask_updrs3)
        updrs2 = apply_mask(updrs2, visit_mask_updrs3)
        moca   = apply_mask(moca,   visit_mask_updrs3)
        sbr    = apply_mask(sbr,    visit_mask_sbr)
        # DaTscan subregions (correlated with sbr_mean)
        def sub(sbr_col, bias):
            s = sbr_col + rng.normal(bias, 0.15, N)
            return s.clip(0.2, 4.5)
        sbr_cau_L = sub(sbr[:,0], 0.30); sbr_cau_R = sub(sbr[:,0],0.28)
        sbr_put_L = sub(sbr[:,0],-0.25); sbr_put_R = sub(sbr[:,0],-0.27)
        # FreeSurfer proxies
        ent_L = (3.2 - 0.005*updrs3[:,0] + rng.normal(0,0.25,N)).clip(1.5,4.5)
        ent_R = ent_L + rng.normal(0, 0.15, N)
        hip_L = (3800 - 8*np.nan_to_num(updrs3[:,0]) + rng.normal(0,250,N)).clip(1800,5200)
        hip_R = hip_L + rng.normal(0, 150, N)
        put_L = (5000 - 5*np.nan_to_num(updrs3[:,0]) + rng.normal(0,300,N)).clip(2500,6500)
        put_R = put_L + rng.normal(0, 180, N)
        # CSF (missingness cohort-dependent)
        csf_asyn = (1000 - 4*np.nan_to_num(updrs3[:,0]) - 150*GBA + rng.normal(0,120,N)).clip(200,2500)
        csf_obs  = rng.binomial(1, np.where(cohort=='Prodromal', 0.7, 0.35), N).astype(bool)
        csf_asyn = np.where(csf_obs, csf_asyn, np.nan)
        # NSD-ISS stage (ordinal 0-6)
        base = np.nan_to_num(updrs3[:,0])/15 + 0.3*GBA
        stage = np.clip(np.round(base + rng.normal(0,0.3,N)),0,6).astype(int)
        stage[cohort=='HC'] = 0
        # Build dataframe
        rows = []
        for i in range(N):
            row = dict(PATNO=10000+i, cohort=cohort[i], age=age[i], sex=sex[i], edu=edu[i],
                       GBA_positive=int(GBA[i]), SAA_positive=int(SAA[i]),
                       BL_UPDRS3=updrs3[i,0], V02_UPDRS3=updrs3[i,1], V04_UPDRS3=updrs3[i,2], V06_UPDRS3=updrs3[i,3],
                       BL_UPDRS1=updrs1[i,0], V02_UPDRS1=updrs1[i,1], V04_UPDRS1=updrs1[i,2], V06_UPDRS1=updrs1[i,3],
                       BL_UPDRS2=updrs2[i,0], V02_UPDRS2=updrs2[i,1], V04_UPDRS2=updrs2[i,2], V06_UPDRS2=updrs2[i,3],
                       BL_MoCA=moca[i,0], V02_MoCA=moca[i,1], V04_MoCA=moca[i,2], V06_MoCA=moca[i,3],
                       BL_SBR_mean=sbr[i,0], V02_SBR_mean=sbr[i,1], V04_SBR_mean=sbr[i,2], V06_SBR_mean=sbr[i,3],
                       SBR_caudate_L=sbr_cau_L[i], SBR_caudate_R=sbr_cau_R[i],
                       SBR_putamen_L=sbr_put_L[i], SBR_putamen_R=sbr_put_R[i],
                       CTh_entorhinal_L=ent_L[i], CTh_entorhinal_R=ent_R[i],
                       GMV_hippocampus_L=hip_L[i], GMV_hippocampus_R=hip_R[i],
                       GMV_putamen_L=put_L[i], GMV_putamen_R=put_R[i],
                       CSF_asyn=csf_asyn[i], NSD_ISS_stage=stage[i],
                       SBR_mean=sbr[i,0])
            rows.append(row)
        master_df = pd.DataFrame(rows)
        print(f'Synthetic cohort built: {master_df.shape}')
    # Ensure required columns exist (safety net)
    for col in ['PATNO','cohort','GBA_positive','SAA_positive']:
        assert col in master_df.columns, f'missing {col}'

    # Deterministic 70/15/15 split by PATNO
    patnos = master_df['PATNO'].values
    rng = np.random.default_rng(SEED)
    idx = rng.permutation(len(patnos))
    n = len(patnos); n_tr = int(0.70*n); n_va = int(0.15*n)
    train_patnos = patnos[idx[:n_tr]]
    val_patnos   = patnos[idx[n_tr:n_tr+n_va]]
    test_patnos  = patnos[idx[n_tr+n_va:]]

    master_df.to_parquet(BLOCK_01A, index=False)
    np.save(SPLIT_TR, train_patnos); np.save(SPLIT_VA, val_patnos); np.save(SPLIT_TE, test_patnos)
    print(f'Splits saved: tr={len(train_patnos)} va={len(val_patnos)} te={len(test_patnos)}')

print('Cohort counts:', master_df['cohort'].value_counts().to_dict())
print('GBA+:', int(master_df['GBA_positive'].sum()), 'SAA+:', int(master_df['SAA_positive'].sum()))

In [ ]:
# ==== Block 01-B — register enriched CSVs (DaTscan subregion, FreeSurfer) if present ====
# With the synthetic cohort, the subregion/FS columns are already inside master_df.
# For real PPMI, join external CSVs here. We handle both cases uniformly.
optional_tables = {
    'DaTscan_subregion':   DATA_RAW/'DaTscan_subregion.csv',
    'Cortical_Thickness':  DATA_RAW/'Cortical_Thickness_CTh.csv',
    'Grey_Matter_Volume':  DATA_RAW/'Grey_Matter_Volume.csv',
    'Volume_ASEG':         DATA_RAW/'Volume_ASEG.csv',
}
for name, path in optional_tables.items():
    if path.exists():
        df_ext = pd.read_csv(path)
        if 'PATNO' in df_ext.columns:
            master_df = master_df.merge(df_ext, on='PATNO', how='left', suffixes=('','_'+name))
            print(f'Merged {name}: +{df_ext.shape[1]-1} cols')
master_df.to_parquet(BLOCK_01A, index=False)
print('Block 01-B done. master_df:', master_df.shape)

In [ ]:
# ==== Block 01-C — column resolution helpers and feature groups ====
def find_col(df, *candidates, default=None):
    '''Return the first column name in df that matches any candidate (case-insensitive substring).
    Falls back to default (which may be None).'''
    cols_lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c is None: continue
        if c in df.columns: return c
        lc = c.lower()
        if lc in cols_lower: return cols_lower[lc]
        # substring fallback
        for k, orig in cols_lower.items():
            if lc in k:
                return orig
    return default

def safe_col(df, name, fill=0.0):
    '''Return np.float32 array; if column absent return zeros + missingness flag.'''
    if name is None or name not in df.columns:
        return np.full(len(df), fill, dtype=np.float32), np.ones(len(df), dtype=np.float32)
    v = df[name].astype(float).values
    miss = np.isnan(v).astype(np.float32)
    v = np.nan_to_num(v, nan=fill).astype(np.float32)
    return v, miss

# Fix 2 — Imaging features: 10-12 spatially distinct, NOT just SBR_mean.
IMAGING_FEATURES = [
    find_col(master_df, 'SBR_caudate_L', 'DATSCAN_CAUDATE_L', default='SBR_caudate_L'),
    find_col(master_df, 'SBR_caudate_R', 'DATSCAN_CAUDATE_R', default='SBR_caudate_R'),
    find_col(master_df, 'SBR_putamen_L', 'DATSCAN_PUTAMEN_L', default='SBR_putamen_L'),
    find_col(master_df, 'SBR_putamen_R', 'DATSCAN_PUTAMEN_R', default='SBR_putamen_R'),
    find_col(master_df, 'SBR_mean', 'BL_SBR_mean', default='SBR_mean'),
    find_col(master_df, 'CTh_entorhinal_L', default='CTh_entorhinal_L'),
    find_col(master_df, 'CTh_entorhinal_R', default='CTh_entorhinal_R'),
    find_col(master_df, 'GMV_hippocampus_L', default='GMV_hippocampus_L'),
    find_col(master_df, 'GMV_hippocampus_R', default='GMV_hippocampus_R'),
    find_col(master_df, 'GMV_putamen_L', default='GMV_putamen_L'),
    find_col(master_df, 'GMV_putamen_R', default='GMV_putamen_R'),
]
CLINICAL_FEATURES  = ['age','sex','edu',
                      find_col(master_df,'BL_UPDRS1'),
                      find_col(master_df,'BL_UPDRS2'),
                      find_col(master_df,'BL_UPDRS3'),
                      find_col(master_df,'BL_MoCA')]
GENETIC_FEATURES   = ['GBA_positive','SAA_positive']
BIOSPECIMEN_FEATURES = [find_col(master_df,'CSF_asyn', default='CSF_asyn')]

# Visit columns
VISIT_PREFIX = ['BL','V02','V04','V06']
VISIT_COLS_U3   = [find_col(master_df,f'{v}_UPDRS3') for v in VISIT_PREFIX]
VISIT_COLS_U1   = [find_col(master_df,f'{v}_UPDRS1') for v in VISIT_PREFIX]
VISIT_COLS_U2   = [find_col(master_df,f'{v}_UPDRS2') for v in VISIT_PREFIX]
VISIT_COLS_MOCA = [find_col(master_df,f'{v}_MoCA')   for v in VISIT_PREFIX]
VISIT_COLS_SBR  = [find_col(master_df,f'{v}_SBR_mean') for v in VISIT_PREFIX]
VISIT_BUNDLES   = [VISIT_COLS_U3, VISIT_COLS_U1, VISIT_COLS_U2, VISIT_COLS_MOCA, VISIT_COLS_SBR]

print('Imaging d =', len(IMAGING_FEATURES))
print('Clinical:', CLINICAL_FEATURES)
print('Genetic :', GENETIC_FEATURES)
print('Bio     :', BIOSPECIMEN_FEATURES)
print('VISIT U3:', VISIT_COLS_U3)

In [ ]:
# ==== Block 01-D — build baseline modality matrices ====
def build_modality_matrix(df, feat_list, scaler=None):
    '''Stack a list of columns into [N, d]. Missingness flags concatenated as extra dims.'''
    cols, miss_cols = [], []
    for f in feat_list:
        v, m = safe_col(df, f)
        cols.append(v); miss_cols.append(m)
    X = np.stack(cols, axis=1).astype(np.float32)
    M = np.stack(miss_cols, axis=1).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0)
    return X, M

X_clin, M_clin = build_modality_matrix(master_df, CLINICAL_FEATURES)
X_gen,  M_gen  = build_modality_matrix(master_df, GENETIC_FEATURES)
X_img,  M_img  = build_modality_matrix(master_df, IMAGING_FEATURES)
X_bio,  M_bio  = build_modality_matrix(master_df, BIOSPECIMEN_FEATURES)

# Concatenate raw + missingness flag to preserve 'missing != zero'
X_clin = np.concatenate([X_clin, M_clin], axis=1)
X_gen  = np.concatenate([X_gen,  M_gen ], axis=1)
X_img  = np.concatenate([X_img,  M_img ], axis=1)
X_bio  = np.concatenate([X_bio,  M_bio ], axis=1)

print('Shapes:', X_clin.shape, X_gen.shape, X_img.shape, X_bio.shape)

np.savez(DATA_PROC/'modalities_unscaled.npz',
         X_clin=X_clin, X_gen=X_gen, X_img=X_img, X_bio=X_bio,
         patnos=master_df['PATNO'].values)

In [ ]:
# ==== Block 01-E — longitudinal visit tensor [N, T, d_visit] ====
def build_visit_tensor(df, bundles):
    '''bundles is a list over visit features; each is a list over T visits of column names.'''
    N = len(df); d = len(bundles); T = CFG['T_MAX']
    x = np.zeros((N, T, d), dtype=np.float32)
    mask = np.zeros((N, T), dtype=np.float32)    # visit observed at ANY feature
    for fi, cols in enumerate(bundles):
        for ti, c in enumerate(cols):
            if c is None or c not in df.columns:
                continue
            v = df[c].astype(float).values
            obs = ~np.isnan(v)
            x[:, ti, fi] = np.nan_to_num(v, nan=0.0).astype(np.float32)
            mask[obs, ti] = 1.0
    return x, mask

X_visit, visit_mask = build_visit_tensor(master_df, VISIT_BUNDLES)
print('Visit tensor:', X_visit.shape, 'mask:', visit_mask.shape,
      'observed frac/visit:', visit_mask.mean(axis=0))
np.save(DATA_PROC/'X_visit.npy', X_visit)
np.save(DATA_PROC/'visit_mask.npy', visit_mask)

In [ ]:
# ==== Block 01-F — LTP Legendre decomposition (3-visit ONLY; Fix 1) ====
# Visits used: BL, V02, V04. V06 is never used because V06_UPDRS3 is the target.
LTP_VISIT_PREFIX = ['BL','V02','V04']
LTP_VISIT_BUNDLES = [VISIT_COLS_U3[:3], VISIT_COLS_U1[:3], VISIT_COLS_U2[:3],
                     VISIT_COLS_MOCA[:3], VISIT_COLS_SBR[:3]]

def legendre_fit_row(y, t):
    '''Fit c0,c1,c2 using only non-NaN entries of y at grid t.
    Returns 3 coefficients; zeros if <2 observations.'''
    obs = ~np.isnan(y)
    if obs.sum() < 2:
        return np.zeros(3, dtype=np.float32)
    t_obs, y_obs = t[obs], y[obs]
    P0 = np.ones_like(t_obs)
    P1 = t_obs
    P2 = 0.5*(3*t_obs**2 - 1)
    Phi = np.stack([P0, P1, P2], axis=1)
    if obs.sum() == 2:
        Phi = Phi[:, :2]
        c, *_ = np.linalg.lstsq(Phi, y_obs, rcond=None)
        return np.concatenate([c, [0.0]]).astype(np.float32)
    c, *_ = np.linalg.lstsq(Phi, y_obs, rcond=None)
    return c.astype(np.float32)

def compute_ltp_features(df, visit_bundles):
    '''Returns [N, 3*len(bundles)] — per-feature (c0, c1, c2).'''
    N = len(df); F_ = len(visit_bundles)
    T3 = len(visit_bundles[0])
    t_grid = np.array([-1.0, 0.0, 1.0], dtype=np.float32)[:T3]
    out = np.zeros((N, 3*F_), dtype=np.float32)
    for fi, cols in enumerate(visit_bundles):
        cols = [c for c in cols if c is not None and c in df.columns]
        if len(cols) < 2: continue
        Y = df[cols].astype(float).values  # [N, T3']
        t = t_grid[:Y.shape[1]]
        for i in range(N):
            out[i, 3*fi:3*fi+3] = legendre_fit_row(Y[i], t)
    return out

X_ltp = compute_ltp_features(master_df, LTP_VISIT_BUNDLES)  # [N, 15]
print('LTP features:', X_ltp.shape,
      ' c1(UPDRS3) mean/std:', X_ltp[:,1].mean().round(3), X_ltp[:,1].std().round(3))
np.save(DATA_PROC/'X_ltp.npy', X_ltp)

In [ ]:
# ==== Block 01-G — IMPE per-node missingness [N, 10] ====
# Modality missingness per patient — fraction of missing values in each block.
def modality_miss_frac(df, feat_list):
    M = np.zeros(len(df), dtype=np.float32)
    if not feat_list: return M
    count = 0
    for f in feat_list:
        if f is None or f not in df.columns: continue
        M += df[f].isna().astype(float).values
        count += 1
    return (M / max(count, 1)).astype(np.float32)

miss_clin = modality_miss_frac(master_df, CLINICAL_FEATURES)
miss_gen  = modality_miss_frac(master_df, GENETIC_FEATURES)
miss_img  = modality_miss_frac(master_df, IMAGING_FEATURES)
miss_bio  = modality_miss_frac(master_df, BIOSPECIMEN_FEATURES)
miss_mod  = np.stack([miss_clin, miss_gen, miss_img, miss_bio], axis=1)  # [N, 4]

# Prespecified node→modality mapping.  Rows = 10 Braak nodes, cols = [clin,gen,img,bio]
NODE_MISS_WEIGHTS = np.array([
    [0.1, 0.0, 0.3, 0.6],   # 0 DMV          — brainstem, CSF heavy
    [0.1, 0.0, 0.4, 0.5],   # 1 LC
    [0.0, 0.0, 1.0, 0.0],   # 2 SNc          — DaTscan
    [0.2, 0.0, 0.5, 0.3],   # 3 Amygdala
    [0.2, 0.0, 0.6, 0.2],   # 4 Hippocampus
    [0.2, 0.0, 0.7, 0.1],   # 5 Entorhinal
    [0.3, 0.0, 0.7, 0.0],   # 6 Temporal
    [0.3, 0.0, 0.7, 0.0],   # 7 Associative
    [0.4, 0.0, 0.6, 0.0],   # 8 Motor
    [0.4, 0.0, 0.6, 0.0],   # 9 Prefrontal
], dtype=np.float32)

node_miss = (miss_mod @ NODE_MISS_WEIGHTS.T).clip(0.0, 1.0).astype(np.float32)   # [N, 10]
print('node_miss shape:', node_miss.shape, ' mean (SNc):', node_miss[:,2].mean().round(3))
np.save(DATA_PROC/'node_miss.npy', node_miss)

In [ ]:
# ==== Block 01-H — hazard labels for NHH (Fix 4: proper right-censoring) ====
# event if UPDRS-III >= 40 at any visit; otherwise censored at LAST OBSERVED visit.
THRESHOLD_UPDRS = 40.0
def compute_hazard_label(row, cols):
    obs_t = [t for t, c in enumerate(cols) if c is not None and c in row.index and pd.notna(row[c])]
    if not obs_t:
        return 0, 0   # no data — censor at t=0
    # event?
    for t in obs_t:
        v = row[cols[t]]
        if pd.notna(v) and v >= THRESHOLD_UPDRS:
            return int(t), 1
    # censored — last observed visit regardless of UPDRS magnitude
    return int(max(obs_t)), 0

event_time = np.zeros(len(master_df), dtype=np.int64)
event_ind  = np.zeros(len(master_df), dtype=np.float32)
for i, (_, row) in enumerate(master_df.iterrows()):
    t_i, d_i = compute_hazard_label(row, VISIT_COLS_U3)
    event_time[i] = t_i; event_ind[i] = d_i

print('Hazard events:', int(event_ind.sum()), '/', len(event_ind),
      ' rate=', f"{event_ind.mean():.2%}")
np.save(DATA_PROC/'event_time.npy', event_time)
np.save(DATA_PROC/'event_ind.npy',  event_ind)

In [ ]:
# ==== Block 01-I — scale features, compute target, save everything ====
from sklearn.preprocessing import StandardScaler

patnos_all = master_df['PATNO'].values
train_mask = np.isin(patnos_all, train_patnos)
val_mask   = np.isin(patnos_all, val_patnos)
test_mask  = np.isin(patnos_all, test_patnos)

def fit_transform_blocks(name, X):
    scaler = StandardScaler().fit(X[train_mask])
    Xs = scaler.transform(X).astype(np.float32)
    Xs = np.nan_to_num(Xs, nan=0.0)
    with open(DATA_PROC/f'scaler_{name}.pkl','wb') as f: pickle.dump(scaler, f)
    return Xs

X_clin_s = fit_transform_blocks('clin', X_clin)
X_gen_s  = fit_transform_blocks('gen',  X_gen)
X_img_s  = fit_transform_blocks('img',  X_img)
X_bio_s  = fit_transform_blocks('bio',  X_bio)
X_ltp_s  = fit_transform_blocks('ltp',  X_ltp)

# Visit tensor: per-feature z-score across population (train only)
def scale_visit(X, mask):
    X_ = X.copy()
    for f in range(X.shape[-1]):
        m = mask[train_mask].astype(bool).flatten()
        vals = X[train_mask,:,f].reshape(-1)[m]
        if vals.std() < 1e-6: continue
        mu, sd = vals.mean(), vals.std()
        X_[..., f] = (X[..., f] - mu) / sd
    return np.nan_to_num(X_, nan=0.0).astype(np.float32)
X_visit_s = scale_visit(X_visit, visit_mask)

# Targets
target_u3 = master_df[VISIT_COLS_U3[3]].astype(float).values    # V06_UPDRS3
target_observed = ~np.isnan(target_u3)
target_u3_norm  = (np.nan_to_num(target_u3, nan=0.0) / CFG['UPDRS_MAX']).astype(np.float32)

target_stage = master_df['NSD_ISS_stage'].astype(int).values if 'NSD_ISS_stage' in master_df.columns else np.zeros(len(master_df), dtype=int)

target_sbr = master_df[IMAGING_FEATURES[4]].astype(float).values if IMAGING_FEATURES[4] in master_df.columns else np.zeros(len(master_df))
target_sbr_observed = ~np.isnan(target_sbr)
target_sbr = np.nan_to_num(target_sbr, nan=0.0).astype(np.float32)

np.savez(DATA_PROC/'features_scaled.npz',
         X_clin=X_clin_s, X_gen=X_gen_s, X_img=X_img_s, X_bio=X_bio_s,
         X_ltp=X_ltp_s,  X_visit=X_visit_s, visit_mask=visit_mask,
         node_miss=node_miss,
         event_time=event_time, event_ind=event_ind,
         target_u3_norm=target_u3_norm, target_observed=target_observed,
         target_stage=target_stage,
         target_sbr=target_sbr, target_sbr_observed=target_sbr_observed,
         patnos=patnos_all,
         GBA=master_df['GBA_positive'].values,
         SAA=master_df['SAA_positive'].values,
         cohort=master_df['cohort'].values.astype(str))
print('Scaled feature bundle saved to', DATA_PROC/'features_scaled.npz')

## Block 02 — Canonical Braak 10-Region Adjacency

In [ ]:
# ==== Block 02 — canonical Braak graph A_braak ====
# Nodes: 0 DMV, 1 LC, 2 SNc, 3 Amygdala, 4 Hippocampus, 5 Entorhinal,
#        6 Temporal, 7 Associative, 8 Motor cortex, 9 Prefrontal
NODE_NAMES = ['DMV','LC','SNc','Amygdala','Hippocampus','Entorhinal',
              'Temporal','Associative','Motor','Prefrontal']
_edges = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),
          (2,5),(3,5),(5,7),(6,8)]
A_braak = np.zeros((10,10), dtype=np.float32)
for a,b in _edges:
    A_braak[a,b] = 1; A_braak[b,a] = 1
np.fill_diagonal(A_braak, 1.0)
np.save(DATA_PROC/'A_braak.npy', A_braak)
A_braak_t = torch.tensor(A_braak, device=DEVICE)
print('Braak adjacency sum:', A_braak.sum(), ' shape:', A_braak.shape)

## Block 03 — Model Definition

`StochasticBraakGraph` is the S-PBGL upgrade: Gaussian posterior over node features, analytical edge variance via bilinear form, KL to N(0,I), missingness-driven variance inflation (κ), and symmetric graph normalisation (Fix 3).

All other modules (LTP encoder, BraakGraphConv = GMP-PBG, BSMTA with position-sensitive bias, EDP head, NSD-ISS head, DaTscan head, NHH survival head) follow.

In [ ]:
# ==== Block 03a — MultiModalEncoder ====
class MultiModalEncoder(nn.Module):
    '''Projects each modality through its own MLP, then cross-modal attention fusion.
    Inputs (shapes):  clin [B, d_clin]   gen [B, d_gen]   img [B, d_img]   bio [B, d_bio]
    Output:           z [B, d_model]
    '''
    def __init__(self, d_clin, d_gen, d_img, d_bio, d_model=128):
        super().__init__()
        self.enc_clin = nn.Sequential(nn.Linear(d_clin, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, d_model))
        self.enc_gen  = nn.Sequential(nn.Linear(d_gen,  128), nn.GELU(), nn.Linear(128, d_model))
        self.enc_img  = nn.Sequential(nn.Linear(d_img,  256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, d_model))
        self.enc_bio  = nn.Sequential(nn.Linear(d_bio,  128), nn.GELU(), nn.Linear(128, d_model))
        self.attn     = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.ln       = nn.LayerNorm(d_model)

    def forward(self, clin, gen, img, bio):
        c = self.enc_clin(clin); g = self.enc_gen(gen); i = self.enc_img(img); b = self.enc_bio(bio)
        seq = torch.stack([c, g, i, b], dim=1)   # [B, 4, d_model]
        fused, _ = self.attn(seq, seq, seq)
        fused = self.ln(fused + seq)
        return fused.mean(dim=1)                 # [B, d_model]

class LTPEncoder(nn.Module):
    '''LTP features [B, 15] → [B, d_model] (additive injection into z).'''
    def __init__(self, d_in=15, d_model=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in,256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256,d_model))
    def forward(self, x_ltp):
        return self.net(x_ltp)
print('03a ok')

In [ ]:
# ==== Block 03b — StochasticBraakGraph (S-PBGL) ====
class StochasticBraakGraph(nn.Module):
    '''S-PBGL: distributional replacement for PersonalizedBraakGraph.
    Learns q(F_i) = N(mu_F, diag(sigma_F^2)) per patient over 10 nodes × d_node features.
    Missingness inflates sigma_F via a learned kappa. Analytical edge uncertainty is
    derived from the bilinear form variance of S_i[j,k] = F_i[j]·F_i[k]^T / sqrt(d_node).

    Outputs:
      A_i       [B, 10, 10]  — mean personalised adjacency, symmetrically normalised.
      u_edge    [B, 10, 10]  — per-edge epistemic uncertainty map.
      kl_graph  scalar       — KL divergence to N(0,I) summed over nodes/dims, batch mean.
    '''
    def __init__(self, A_braak, d_model=128, n_nodes=10, d_node=8,
                 braak_eps=0.05, scale_init=0.1, kappa_init=0.5,
                 logvar_clip=(-4.0, 2.0)):
        super().__init__()
        self.register_buffer('A_braak', A_braak.float())
        self.n_nodes = n_nodes; self.d_node = d_node
        self.eps = braak_eps; self.logvar_clip = logvar_clip

        self.mu_proj     = nn.Sequential(nn.Linear(d_model, n_nodes*d_node), nn.Tanh())
        self.logvar_proj = nn.Linear(d_model, n_nodes*d_node)

        self.kappa = nn.Parameter(torch.tensor(float(kappa_init)))
        self.scale = nn.Parameter(torch.tensor(float(scale_init)))

    def forward(self, z, node_miss=None):
        B = z.shape[0]
        # Step 1 — mean and log-variance of node features
        mu_F     = self.mu_proj(z).reshape(B, self.n_nodes, self.d_node)            # [B,10,8]
        logvar_F = self.logvar_proj(z).reshape(B, self.n_nodes, self.d_node)        # [B,10,8]
        logvar_F = logvar_F.clamp(self.logvar_clip[0], self.logvar_clip[1])
        sigma_F  = torch.exp(0.5 * logvar_F)                                        # [B,10,8]

        # Step 2 — IMPE: inflate sigma at nodes whose supporting data is missing
        if node_miss is not None:
            infl    = torch.exp(self.kappa * node_miss).unsqueeze(-1)               # [B,10,1]
            sigma_F = sigma_F * infl

        # Step 3 — reparameterisation during training; mean at inference
        if self.training:
            eps = torch.randn_like(sigma_F)
            F_i = mu_F + sigma_F * eps
        else:
            F_i = mu_F

        # Step 4 — mean adjacency with Braak masking + SYMMETRIC normalisation (Fix 3)
        A_prior = (self.A_braak + self.eps).unsqueeze(0)                            # [1,10,10]
        S_i     = torch.bmm(F_i, F_i.transpose(1,2)) / (self.d_node ** 0.5)         # [B,10,10]
        A_raw   = torch.sigmoid(self.scale * S_i) * A_prior
        d       = A_raw.sum(dim=-1, keepdim=True).clamp(min=1e-6)                   # [B,10,1]
        d_inv_sqrt = d.pow(-0.5)
        A_i     = d_inv_sqrt * A_raw * d_inv_sqrt.transpose(1,2)                    # [B,10,10]

        # Step 5 — analytical edge uncertainty (bilinear form variance + delta method)
        mu_norm_sq    = (mu_F    ** 2).sum(-1)                                      # [B,10]
        sigma_norm_sq = (sigma_F ** 2).sum(-1)                                      # [B,10]
        var_S = (sigma_norm_sq.unsqueeze(2) * mu_norm_sq.unsqueeze(1)
               + mu_norm_sq.unsqueeze(2)    * sigma_norm_sq.unsqueeze(1)
               + sigma_norm_sq.unsqueeze(2) * sigma_norm_sq.unsqueeze(1)) / self.d_node  # [B,10,10]
        S_mean = torch.bmm(mu_F, mu_F.transpose(1,2)) / (self.d_node ** 0.5)        # [B,10,10]
        p_bar  = torch.sigmoid(self.scale * S_mean) * A_prior                       # [B,10,10]
        dp_dS  = (self.scale ** 2) * (p_bar ** 2) * ((1 - p_bar) ** 2)              # [B,10,10]
        u_edge = dp_dS * var_S * (A_prior ** 2)                                     # [B,10,10]

        # Step 6 — KL(q||p=N(0,I))
        kl = 0.5 * (mu_F**2 + sigma_F**2 - 1 - logvar_F).sum(dim=[1,2]).mean()

        return A_i, u_edge, kl

    def graph_reg_loss(self, A_i):
        '''Frobenius distance of the learned graph to the canonical Braak prior.'''
        tgt = self.A_braak.unsqueeze(0).expand_as(A_i)
        return F.mse_loss(A_i, tgt)
print('03b ok (S-PBGL)')

In [ ]:
# ==== Block 03c — BraakGraphConv (GMP-PBG) ====
class BraakGraphConv(nn.Module):
    '''Two rounds of graph convolution on A_i with imaging features initialising nodes.
    Inputs:  x_img [B, d_img], A_i [B, 10, 10].
    Output:  g_i [B, d_model] — injected additively into patient embedding.
    '''
    def __init__(self, d_img, d_model=128, n_nodes=10, d_hidden=16):
        super().__init__()
        self.n_nodes = n_nodes; self.d_hidden = d_hidden
        self.init_proj = nn.Linear(d_img, n_nodes * d_hidden)
        self.W1      = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W1_self = nn.Linear(d_hidden, d_hidden)
        self.W2      = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W2_self = nn.Linear(d_hidden, d_hidden)
        self.ln1 = nn.LayerNorm(d_hidden); self.ln2 = nn.LayerNorm(d_hidden)
        self.out = nn.Linear(d_hidden, d_model)

    def forward(self, x_img, A_i):
        B = x_img.shape[0]
        H0 = self.init_proj(x_img).reshape(B, self.n_nodes, self.d_hidden)    # [B,10,dh]
        H1 = F.gelu(self.ln1(torch.bmm(A_i, self.W1(H0)) + self.W1_self(H0)))
        H2 = F.gelu(self.ln2(torch.bmm(A_i, self.W2(H1)) + self.W2_self(H1)))
        g  = H2.mean(dim=1)                                                   # [B, dh]
        return self.out(g)                                                    # [B, d_model]
print('03c ok')

In [ ]:
# ==== Block 03d — BSMTA (position-sensitive Braak pathway attention) ====
class BraakStructuredAttentionLayer(nn.Module):
    '''Single transformer layer with a per-head learnable [T x n_nodes] query matrix.
    The Braak bias B_i^(h)[t,t'] = <C_i^(h)[t,:], C_i^(h)[t',:]> / sqrt(n_nodes)
    where C_i^(h) = Q^(h)[:T,:] @ A_i is patient-specific via A_i.'''
    def __init__(self, d_model=128, n_heads=8, n_nodes=10, T_max=4, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model=d_model; self.n_heads=n_heads; self.d_h = d_model//n_heads
        self.n_nodes=n_nodes; self.T_max=T_max
        self.qkv  = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.temporal_node_queries = nn.Parameter(torch.randn(n_heads, T_max, n_nodes)*0.02)
        self.head_scale = nn.Parameter(torch.ones(n_heads) * 0.1)
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.GELU(),
                                 nn.Linear(4*d_model, d_model), nn.Dropout(dropout))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, A_i, visit_mask=None):
        B, T, d = x.shape
        assert T <= self.T_max
        qkv = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q,k,v = [t.reshape(B, T, self.n_heads, self.d_h).transpose(1,2) for t in qkv]  # [B,H,T,dh]
        scores = (q @ k.transpose(-1,-2)) / (self.d_h ** 0.5)                           # [B,H,T,T]
        Q_h = self.temporal_node_queries[:, :T, :]                                      # [H,T,10]
        C = torch.einsum('htn,bnm->bhtm', Q_h, A_i)                                     # [B,H,T,10]
        B_bias = torch.einsum('bhtn,bhsn->bhts', C, C) / (self.n_nodes ** 0.5)         # [B,H,T,T]
        scores = scores + self.head_scale.view(1,-1,1,1) * B_bias
        if visit_mask is not None:
            m = visit_mask.unsqueeze(1).unsqueeze(1)                                    # [B,1,1,T]
            scores = scores.masked_fill(m == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.drop(attn)
        out = (attn @ v).transpose(1,2).reshape(B, T, d)
        out = x + self.proj(out)
        out = out + self.mlp(self.ln2(out))
        return out, attn

class BSMTA(nn.Module):
    def __init__(self, d_model=128, d_visit=5, n_heads=8, n_nodes=10, T_max=4,
                 n_layers=2, dropout=0.1):
        super().__init__()
        self.visit_embed = nn.Linear(d_visit, d_model)
        self.pos_embed   = nn.Parameter(torch.randn(T_max, d_model)*0.02)
        self.layers = nn.ModuleList([
            BraakStructuredAttentionLayer(d_model, n_heads, n_nodes, T_max, dropout)
            for _ in range(n_layers)])
    def forward(self, x_visit, A_i, z_ctx=None, visit_mask=None):
        B, T, _ = x_visit.shape
        h = self.visit_embed(x_visit) + self.pos_embed[:T].unsqueeze(0)
        if z_ctx is not None:
            h = h + z_ctx.unsqueeze(1)
        attns = []
        for layer in self.layers:
            h, a = layer(h, A_i, visit_mask)
            attns.append(a)
        # last-visit representation (select last observed visit per patient)
        if visit_mask is not None:
            last_idx = (visit_mask.sum(dim=1).long() - 1).clamp(min=0)
            idx = last_idx.view(-1,1,1).expand(-1,1,h.shape[-1])
            h_last = torch.gather(h, 1, idx).squeeze(1)
        else:
            h_last = h[:, -1, :]
        if z_ctx is not None:
            h_last = h_last + z_ctx
        return h_last, attns
print('03d ok')

In [ ]:
# ==== Block 03e — Heads ====
class EDPHead(nn.Module):
    '''Normal-Inverse-Gamma evidential regression head → (γ, ν, α, β).'''
    def __init__(self, d_in=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 128), nn.GELU(), nn.Linear(128, 4))
    def forward(self, h):
        out = self.net(h)
        gamma = out[:, 0]
        nu    = F.softplus(out[:, 1]) + 1e-6
        alpha = F.softplus(out[:, 2]) + 1.0 + 1e-6
        beta  = F.softplus(out[:, 3]) + 1e-6
        return gamma, nu, alpha, beta
    @staticmethod
    def nig_nll(y, gamma, nu, alpha, beta, lam=0.01):
        two_beta = 2.0 * beta * (1 + nu)
        nll = 0.5 * torch.log(math.pi / nu) \
            - alpha * torch.log(two_beta) \
            + (alpha + 0.5) * torch.log(nu * (y - gamma)**2 + two_beta) \
            + torch.lgamma(alpha) - torch.lgamma(alpha + 0.5)
        reg = torch.abs(y - gamma) * (2 * nu + alpha)
        return nll.mean() + lam * reg.mean()
    @staticmethod
    def uncertainty(nu, alpha, beta):
        u_epi = beta / (nu * (alpha - 1).clamp(min=1e-6))
        u_ale = beta / (alpha - 1).clamp(min=1e-6)
        return u_epi, u_ale

class StageHead(nn.Module):
    def __init__(self, d_in=128, n_stages=7):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in,128), nn.GELU(), nn.Linear(128, n_stages))
    def forward(self, h): return self.net(h)

class DaTscanHead(nn.Module):
    def __init__(self, d_in=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in,128), nn.GELU(), nn.Linear(128,1))
    def forward(self, h): return self.net(h).squeeze(-1)

class NeuralHazardHead(nn.Module):
    '''Discrete-time hazard head over T bins. Vectorised NLL (Fix 7).'''
    def __init__(self, d_in=128, n_bins=4, d_emb=8):
        super().__init__()
        self.n_bins = n_bins
        self.t_emb = nn.Embedding(n_bins, d_emb)
        self.mlp  = nn.Sequential(nn.Linear(d_in + d_emb, 64), nn.GELU(), nn.Linear(64, 1))
    def forward(self, h):
        B = h.shape[0]
        t_ids = torch.arange(self.n_bins, device=h.device)
        t_e   = self.t_emb(t_ids)                              # [T, d_emb]
        h_e   = h.unsqueeze(1).expand(-1, self.n_bins, -1)     # [B,T,d]
        t_e   = t_e.unsqueeze(0).expand(B, -1, -1)             # [B,T,d_emb]
        logits= self.mlp(torch.cat([h_e, t_e], dim=-1)).squeeze(-1)  # [B,T]
        return logits
    def loss(self, logits, event_time, event_ind):
        B, T = logits.shape
        tr = torch.arange(T, device=logits.device).unsqueeze(0)
        before = (tr < event_time.unsqueeze(1)).float()
        at_T   = (tr == event_time.unsqueeze(1)).float()
        term1  = (F.logsigmoid(-logits) * before).sum(1)
        term2  = (F.logsigmoid( logits) * at_T * event_ind.unsqueeze(1)).sum(1)
        return -(term1 + term2).mean()
    @staticmethod
    def survival_curve(logits):
        h = torch.sigmoid(logits)                       # [B,T]
        S = torch.cumprod(1.0 - h, dim=1)
        return S

class ProjectionHead(nn.Module):
    def __init__(self, d_in=128, d_out=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_in), nn.GELU(), nn.Linear(d_in, d_out))
    def forward(self, z): return F.normalize(self.net(z), dim=-1)

print('03e ok')

In [ ]:
# ==== Block 03f — BrainFormerPD end-to-end ====
class BrainFormerPDv21(nn.Module):
    '''Full model. encode_patient returns (z, A_i, u_edge, kl_graph) so the two-source
    uncertainty story and S-PBGL regulariser are accessible at every call site.'''
    def __init__(self, d_clin, d_gen, d_img, d_bio, d_ltp=15,
                 d_model=128, d_visit=5, n_heads=8, n_nodes=10, T_max=4,
                 n_layers=2, n_stages=7, n_bins=4, A_braak=None, cfg=CFG):
        super().__init__()
        self.d_img = d_img
        self.mod_enc = MultiModalEncoder(d_clin, d_gen, d_img, d_bio, d_model)
        self.ltp_enc = LTPEncoder(d_ltp, d_model)
        self.graph   = StochasticBraakGraph(A_braak, d_model, n_nodes,
                                            cfg['d_node'], cfg['braak_eps'],
                                            cfg['graph_scale_init'], cfg['kappa_init'],
                                            cfg['logvar_clip'])
        self.gcn     = BraakGraphConv(d_img, d_model, n_nodes)
        self.temporal= BSMTA(d_model, d_visit, n_heads, n_nodes, T_max, n_layers)
        self.h_edp   = EDPHead(d_model)
        self.h_stg   = StageHead(d_model, n_stages)
        self.h_sbr   = DaTscanHead(d_model)
        self.h_nhh   = NeuralHazardHead(d_model, n_bins)
        self.proj    = ProjectionHead(d_model, 64)

    def encode_patient(self, batch):
        z = self.mod_enc(batch['clin'], batch['gen'], batch['img'], batch['bio'])
        z = z + self.ltp_enc(batch['ltp'])
        A_i, u_edge, kl = self.graph(z, batch.get('node_miss'))
        z = z + self.gcn(batch['img_raw'], A_i)
        return z, A_i, u_edge, kl

    def forward(self, batch):
        z, A_i, u_edge, kl = self.encode_patient(batch)
        h_last, attns = self.temporal(batch['visit'], A_i, z_ctx=z,
                                       visit_mask=batch.get('visit_mask'))
        gamma, nu, alpha, beta = self.h_edp(h_last)
        stage_logits = self.h_stg(h_last)
        sbr_pred     = self.h_sbr(h_last)
        hazard_logits= self.h_nhh(h_last)
        proj_z       = self.proj(z)
        return dict(gamma=gamma, nu=nu, alpha=alpha, beta=beta,
                    stage_logits=stage_logits, sbr_pred=sbr_pred,
                    hazard_logits=hazard_logits, z=z, proj=proj_z,
                    A_i=A_i, u_edge=u_edge, kl_graph=kl,
                    h_last=h_last, attns=attns)
print('03f ok (BrainFormerPDv21)')

## Block 04 — Dataset and DataLoaders

In [ ]:
# ==== Block 04 — PPMIDatasetV21 + loaders ====
_data = np.load(DATA_PROC/'features_scaled.npz', allow_pickle=True)

class PPMIDatasetV21(Dataset):
    def __init__(self, indices, arrays):
        self.idx = np.asarray(indices)
        self.a   = arrays
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        k = int(self.idx[i])
        item = dict(
            clin     = torch.from_numpy(np.nan_to_num(self.a['X_clin'][k], nan=0.0)),
            gen      = torch.from_numpy(np.nan_to_num(self.a['X_gen'][k],  nan=0.0)),
            img      = torch.from_numpy(np.nan_to_num(self.a['X_img'][k],  nan=0.0)),
            bio      = torch.from_numpy(np.nan_to_num(self.a['X_bio'][k],  nan=0.0)),
            ltp      = torch.from_numpy(np.nan_to_num(self.a['X_ltp'][k],  nan=0.0)),
            visit    = torch.from_numpy(np.nan_to_num(self.a['X_visit'][k], nan=0.0)),
            visit_mask=torch.from_numpy(self.a['visit_mask'][k].astype(np.float32)),
            node_miss= torch.from_numpy(self.a['node_miss'][k].astype(np.float32)),
            img_raw  = torch.from_numpy(np.nan_to_num(self.a['X_img'][k], nan=0.0)),
            y_updrs  = torch.tensor(self.a['target_u3_norm'][k], dtype=torch.float32),
            y_obs    = torch.tensor(float(self.a['target_observed'][k]), dtype=torch.float32),
            y_stage  = torch.tensor(int(self.a['target_stage'][k]), dtype=torch.long),
            y_sbr    = torch.tensor(float(self.a['target_sbr'][k]), dtype=torch.float32),
            y_sbr_obs= torch.tensor(float(self.a['target_sbr_observed'][k]), dtype=torch.float32),
            event_time=torch.tensor(int(self.a['event_time'][k]), dtype=torch.long),
            event_ind =torch.tensor(float(self.a['event_ind'][k]), dtype=torch.float32),
            GBA      =torch.tensor(int(self.a['GBA'][k]), dtype=torch.long),
            SAA      =torch.tensor(int(self.a['SAA'][k]), dtype=torch.long),
            patno    =torch.tensor(int(self.a['patnos'][k]), dtype=torch.long),
        )
        return item

patnos_all = _data['patnos']
idx_tr = np.where(np.isin(patnos_all, train_patnos))[0]
idx_va = np.where(np.isin(patnos_all, val_patnos))[0]
idx_te = np.where(np.isin(patnos_all, test_patnos))[0]

ds_tr = PPMIDatasetV21(idx_tr, _data)
ds_va = PPMIDatasetV21(idx_va, _data)
ds_te = PPMIDatasetV21(idx_te, _data)

loader_tr = DataLoader(ds_tr, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, drop_last=True)
loader_va = DataLoader(ds_va, batch_size=CFG['batch_size'], shuffle=False, num_workers=2)
loader_te = DataLoader(ds_te, batch_size=CFG['batch_size'], shuffle=False, num_workers=2)

# Derive modality dimensions
D_CLIN = _data['X_clin'].shape[1]
D_GEN  = _data['X_gen'].shape[1]
D_IMG  = _data['X_img'].shape[1]
D_BIO  = _data['X_bio'].shape[1]
D_LTP  = _data['X_ltp'].shape[1]
D_VISIT= _data['X_visit'].shape[2]
N_STAGES = int(_data['target_stage'].max() + 1)
print(f'dims  clin={D_CLIN} gen={D_GEN} img={D_IMG} bio={D_BIO} ltp={D_LTP} visit={D_VISIT} stages={N_STAGES}')
print(f'train/val/test = {len(ds_tr)}/{len(ds_va)}/{len(ds_te)}')

## Block 05 — GBA-BCSD Contrastive Pre-training

Supervised contrastive (Khosla et al. 2020). Positives = same GBA status with |ΔUPDRS| < 5. Negatives = different GBA status. Hard negatives = different GBA + similar UPDRS.

In [ ]:
# ==== Block 05a — supervised contrastive loss ====
def supervised_contrastive_loss(proj, labels, updrs, tau=0.1, hard_eta=1.0, sigma=5.0):
    '''proj [B, d] L2-normed. labels [B] (GBA), updrs [B].
    Positives: same GBA AND |ΔUPDRS|<5.  Hard negatives: different GBA with small UPDRS gap.'''
    B = proj.shape[0]
    sim = proj @ proj.t() / tau                                              # [B,B]
    mask_self = torch.eye(B, device=proj.device, dtype=torch.bool)
    sim = sim.masked_fill(mask_self, float('-inf'))
    dU = (updrs.unsqueeze(0) - updrs.unsqueeze(1)).abs()                     # [B,B]
    same_g = (labels.unsqueeze(0) == labels.unsqueeze(1))                    # [B,B]
    pos_mask = same_g & (dU < sigma) & (~mask_self)
    # Skip rows with no positives
    valid = pos_mask.any(dim=1)
    if valid.sum() < 2:
        return proj.sum() * 0.0
    # Hard-negative weight: different GBA with similar UPDRS
    diff_g = (~same_g)
    w_hard = 1.0 + hard_eta * torch.exp(-dU / (sigma*2.0)) * diff_g.float()   # [B,B]
    logits = sim + torch.log(w_hard.clamp(min=1e-6))
    logits = logits[valid]                                                   # [B',B]
    pos_mask_v = pos_mask[valid]
    log_prob  = logits - torch.logsumexp(logits, dim=1, keepdim=True)        # normalise
    mean_log_prob_pos = (log_prob * pos_mask_v.float()).sum(1) / pos_mask_v.float().sum(1).clamp(min=1.0)
    return -mean_log_prob_pos.mean()
print('05a ok (supervised contrastive)')

In [ ]:
# ==== Block 05b — pre-training loop (resume-aware) ====
BLOCK_05_CKPT = CHECKPOINTS/'block_05_pretrain.pt'
SKIP_05 = BLOCK_05_CKPT.exists()

def move(batch, dev):
    return {k:(v.to(dev) if torch.is_tensor(v) else v) for k,v in batch.items()}

A_braak_t = torch.tensor(A_braak, device=DEVICE)

def build_model():
    m = BrainFormerPDv21(D_CLIN, D_GEN, D_IMG, D_BIO, d_ltp=D_LTP,
                         d_model=CFG['d_model'], d_visit=D_VISIT,
                         n_heads=CFG['n_heads'], n_nodes=CFG['n_nodes'],
                         T_max=CFG['T_MAX'], n_layers=CFG['n_tf_layers'],
                         n_stages=max(N_STAGES, 7), n_bins=CFG['T_MAX'],
                         A_braak=A_braak_t, cfg=CFG).to(DEVICE)
    return m

model = build_model()
print('params:', sum(p.numel() for p in model.parameters())/1e6, 'M')

if SKIP_05:
    ckpt = torch.load(BLOCK_05_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    print('Resumed pre-trained encoder from block_05_pretrain.pt')
else:
    set_seed()
    enc_params = list(model.mod_enc.parameters()) + list(model.ltp_enc.parameters()) \
               + list(model.graph.parameters()) + list(model.gcn.parameters()) \
               + list(model.proj.parameters())
    opt = torch.optim.AdamW(enc_params, lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    pbar = tqdm(range(CFG['pretrain_epochs']), desc='pretrain')
    for epoch in pbar:
        model.train()
        losses = []
        for batch in loader_tr:
            batch = move(batch, DEVICE)
            z, A_i, u_edge, kl = model.encode_patient(batch)
            proj = model.proj(z)
            updrs_proxy = batch['clin'][:, 5] if D_CLIN > 5 else batch['y_updrs']
            loss_c = supervised_contrastive_loss(proj, batch['GBA'], updrs_proxy,
                                                 tau=0.1, hard_eta=1.5, sigma=5.0)
            anneal = min((epoch+1)/CFG['kl_anneal_epochs'], 1.0) * CFG['kl_max']
            loss = loss_c + anneal * kl + CFG['w_graph'] * model.graph.graph_reg_loss(A_i)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(enc_params, CFG['grad_clip'])
            opt.step()
            losses.append(float(loss.item()))
        pbar.set_postfix(loss=np.mean(losses))
    torch.save(dict(block='BLOCK_05_PRETRAIN',
                    model_state=model.state_dict(),
                    optimizer_state=opt.state_dict(),
                    metadata=dict(epochs=CFG['pretrain_epochs'])),
               BLOCK_05_CKPT)
    print('Pre-training done; checkpoint saved.')

## Block 06 — Full Multi-Task Training (150 epochs)

Loss: `L = w_updrs·NIG + w_stage·CE + w_datscan·MSE + w_graph·graph_reg + w_hazard·NHH + anneal_kl·KL`.
UG-HEM sample weighting starts at `epoch ≥ ughem_warmup` using NIG epistemic uncertainty.

In [ ]:
# ==== Block 06a — UG-HEM weight helper ====
def ughem_weight(u_epi, gamma=0.5):
    '''Return soft-clipped weights in [1-gamma, 1+gamma] with stop-gradient.'''
    with torch.no_grad():
        u_norm = u_epi / (u_epi.mean() + 1e-6)
        w = 1.0 + gamma * torch.tanh(u_norm - 1.0)
        return w.detach()
print('06a ok')

In [ ]:
# ==== Block 06b — class weights for staging ====
from sklearn.utils.class_weight import compute_class_weight
y_stage_tr = _data['target_stage'][idx_tr].astype(int)
classes = np.unique(y_stage_tr)
cw = compute_class_weight('balanced', classes=classes, y=y_stage_tr)
class_weights = torch.ones(max(N_STAGES, 7), device=DEVICE)
for c, w in zip(classes, cw): class_weights[int(c)] = float(w)
print('stage class weights:', class_weights.cpu().numpy().round(2))

In [ ]:
# ==== Block 06c — full training loop ====
BLOCK_06_CKPT      = CHECKPOINTS/'block_06_full.pt'
BLOCK_06_BEST_CKPT = CHECKPOINTS/'block_06_best.pt'

def build_loss(outs, batch, epoch, class_weights):
    '''Assemble the combined multi-task loss and a diagnostics dict.'''
    y_obs = batch['y_obs']; y = batch['y_updrs']
    gamma, nu, alpha, beta = outs['gamma'], outs['nu'], outs['alpha'], outs['beta']
    # NIG per-sample
    two_beta = 2.0 * beta * (1 + nu)
    nll_each = 0.5*torch.log(math.pi/nu) - alpha*torch.log(two_beta) \
             + (alpha+0.5)*torch.log(nu*(y-gamma)**2 + two_beta) \
             + torch.lgamma(alpha) - torch.lgamma(alpha+0.5)
    reg_each = torch.abs(y - gamma) * (2*nu + alpha)
    nig_loss_each = nll_each + 0.01 * reg_each
    # UG-HEM reweighting
    u_epi = beta / (nu * (alpha - 1).clamp(min=1e-6))
    if epoch >= CFG['ughem_warmup']:
        w = ughem_weight(u_epi, CFG['ughem_gamma'])
    else:
        w = torch.ones_like(u_epi)
    L_updrs = (nig_loss_each * w * y_obs).sum() / y_obs.sum().clamp(min=1)
    # Stage
    L_stage = F.cross_entropy(outs['stage_logits'], batch['y_stage'], weight=class_weights)
    # DaTscan (masked)
    sbr_err = ((outs['sbr_pred'] - batch['y_sbr'])**2) * batch['y_sbr_obs']
    L_sbr   = sbr_err.sum() / batch['y_sbr_obs'].sum().clamp(min=1)
    # Hazard
    L_haz   = NeuralHazardHead.loss(None, outs['hazard_logits'], batch['event_time'], batch['event_ind'])
    # Graph reg + KL
    L_graph = model.graph.graph_reg_loss(outs['A_i'])
    kl_w    = min((epoch+1)/CFG['kl_anneal_epochs'], 1.0) * CFG['kl_max']
    L_kl    = outs['kl_graph']
    total = (CFG['w_updrs']*L_updrs + CFG['w_stage']*L_stage + CFG['w_datscan']*L_sbr
             + CFG['w_hazard']*L_haz + CFG['w_graph']*L_graph + kl_w*L_kl)
    return total, dict(updrs=L_updrs.item(), stage=L_stage.item(), sbr=L_sbr.item(),
                       haz=L_haz.item(), graph=L_graph.item(), kl=L_kl.item(), kl_w=kl_w)

def eval_r2(loader):
    model.eval()
    ys, ps, obs = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = move(batch, DEVICE)
            outs  = model(batch)
            ys.append(batch['y_updrs'].cpu().numpy())
            ps.append(outs['gamma'].cpu().numpy())
            obs.append(batch['y_obs'].cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps); m = np.concatenate(obs).astype(bool)
    if m.sum() < 10: return float('nan')
    y_r = y[m] * CFG['UPDRS_MAX']; p_r = p[m] * CFG['UPDRS_MAX']
    ss_res = ((y_r - p_r) ** 2).sum(); ss_tot = ((y_r - y_r.mean()) ** 2).sum()
    return 1.0 - ss_res / ss_tot

SKIP_06 = BLOCK_06_CKPT.exists()
if SKIP_06:
    ckpt = torch.load(BLOCK_06_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    print('Resumed full-training checkpoint.')
else:
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)
    best_r2 = -1e9
    history = []
    pbar = tqdm(range(CFG['train_epochs']), desc='train')
    for epoch in pbar:
        model.train()
        total_loss = []
        for batch in loader_tr:
            batch = move(batch, DEVICE)
            outs  = model(batch)
            loss, diag = build_loss(outs, batch, epoch, class_weights)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            opt.step()
            total_loss.append(float(loss.item()))
        sched.step()
        r2_va = eval_r2(loader_va)
        history.append(dict(epoch=epoch, loss=float(np.mean(total_loss)), r2_val=float(r2_va)))
        pbar.set_postfix(loss=np.mean(total_loss), r2_val=r2_va)
        if not np.isnan(r2_va) and r2_va > best_r2:
            best_r2 = r2_va
            torch.save(dict(block='BLOCK_06_BEST',
                            model_state=model.state_dict(),
                            optimizer_state=opt.state_dict(),
                            metadata=dict(epoch=epoch, r2_val=r2_va)),
                       BLOCK_06_BEST_CKPT)
        # periodic checkpoint every 10 epochs
        if (epoch+1) % 10 == 0:
            torch.save(dict(block='BLOCK_06_FULL',
                            model_state=model.state_dict(),
                            optimizer_state=opt.state_dict(),
                            metadata=dict(epoch=epoch, history=history)),
                       BLOCK_06_CKPT)
    # final save
    torch.save(dict(block='BLOCK_06_FULL',
                    model_state=model.state_dict(),
                    optimizer_state=opt.state_dict(),
                    metadata=dict(epoch=epoch, history=history, best_r2_val=best_r2)),
               BLOCK_06_CKPT)
    # restore best
    if BLOCK_06_BEST_CKPT.exists():
        best = torch.load(BLOCK_06_BEST_CKPT, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best['model_state'])
        print('Loaded best checkpoint, r2_val=', best['metadata']['r2_val'])

## Block 07 — Primary Evaluation

Run inference on the test set, persist every per-patient output (incl. `A_i`, `u_edge`, `z`), and compute primary UPDRS-III R² on the **PD cohort**, secondary on Prodromal, plus NSD-ISS Weighted F1, DaTscan Pearson r, and NHH concordance index.

In [ ]:

# ==== Block 07 — primary evaluation ====
from sklearn.metrics import r2_score, mean_absolute_error, f1_score
from scipy.stats import pearsonr, spearmanr

BLOCK_07_CKPT = RESULTS / 'test_outputs.npz'
SKIP_07 = BLOCK_07_CKPT.exists()

def run_inference(loader, model):
    model.eval()
    keys = ['gamma','nu','alpha','beta','stage_logits','sbr','hazard','A','u_edge','z','proj',
            'y_u3','y_obs','y_stage','y_sbr','sbr_obs','GBA','SAA','patno','event_time','event_ind']
    out = {k: [] for k in keys}
    with torch.no_grad():
        for batch in loader:
            b = move(batch, DEVICE)
            r = model(b)
            out['gamma'].append(r['gamma'].cpu().numpy())
            out['nu'].append(r['nu'].cpu().numpy())
            out['alpha'].append(r['alpha'].cpu().numpy())
            out['beta'].append(r['beta'].cpu().numpy())
            out['stage_logits'].append(r['stage_logits'].cpu().numpy())
            out['sbr'].append(r['sbr_pred'].cpu().numpy())
            out['hazard'].append(r['hazard_logits'].cpu().numpy())
            out['A'].append(r['A_i'].cpu().numpy())
            out['u_edge'].append(r['u_edge'].cpu().numpy())
            out['z'].append(r['z'].cpu().numpy())
            out['proj'].append(r['proj'].cpu().numpy())
            out['y_u3'].append(b['y_updrs'].cpu().numpy())
            out['y_obs'].append(b['y_obs'].cpu().numpy())
            out['y_stage'].append(b['y_stage'].cpu().numpy())
            out['y_sbr'].append(b['y_sbr'].cpu().numpy())
            out['sbr_obs'].append(b['y_sbr_obs'].cpu().numpy())
            out['GBA'].append(b['GBA'].cpu().numpy())
            out['SAA'].append(b['SAA'].cpu().numpy())
            out['patno'].append(b['patno'].cpu().numpy())
            out['event_time'].append(b['event_time'].cpu().numpy())
            out['event_ind'].append(b['event_ind'].cpu().numpy())
    return {k: np.concatenate(v, axis=0) for k, v in out.items()}

if SKIP_07:
    loaded = np.load(BLOCK_07_CKPT, allow_pickle=True)
    test_out = {k: loaded[k] for k in loaded.files}
    print('Resumed test_outputs from checkpoint.')
else:
    test_out = run_inference(loader_te, model)
    np.savez(BLOCK_07_CKPT, **test_out)
    print('Saved test_outputs.')

cohort_by_patno = dict(zip(master_df['PATNO'].values, master_df['cohort'].values))
cohort_test = np.array([cohort_by_patno.get(int(p), 'Unknown') for p in test_out['patno']])
pd_mask  = cohort_test == 'PD'
pro_mask = cohort_test == 'Prodromal'

def metrics(y_norm, p_norm, obs):
    m = obs.astype(bool)
    if m.sum() < 10: return None
    y = y_norm[m] * CFG['UPDRS_MAX']; p = p_norm[m] * CFG['UPDRS_MAX']
    return dict(R2=float(r2_score(y, p)), MAE=float(mean_absolute_error(y, p)), n=int(m.sum()))

metr_all = metrics(test_out['y_u3'], test_out['gamma'], test_out['y_obs'])
metr_pd  = metrics(test_out['y_u3'], test_out['gamma'], test_out['y_obs'] * pd_mask.astype(float))
metr_pro = metrics(test_out['y_u3'], test_out['gamma'], test_out['y_obs'] * pro_mask.astype(float))

stage_pred = test_out['stage_logits'].argmax(axis=1)
stage_f1_w = f1_score(test_out['y_stage'], stage_pred, average='weighted')
stage_f1_m = f1_score(test_out['y_stage'], stage_pred, average='macro')

sbr_obs = test_out['sbr_obs'].astype(bool)
if sbr_obs.sum() >= 10:
    r_dat, p_dat = pearsonr(test_out['y_sbr'][sbr_obs], test_out['sbr'][sbr_obs])
    rs_dat, ps_dat = spearmanr(test_out['y_sbr'][sbr_obs], test_out['sbr'][sbr_obs])
else:
    r_dat, p_dat = rs_dat, ps_dat = float('nan'), float('nan')

# NHH concordance index (Harrell's C)
def c_index(event_time, event_ind, risk, max_pairs=2_000_000):
    n = len(event_time); num = den = 0.0; cnt = 0
    for i in range(n):
        ei, ti = event_ind[i], event_time[i]
        if ei != 1: continue
        for j in range(n):
            if i == j: continue
            if ti < event_time[j]:
                den += 1
                if risk[i] > risk[j]: num += 1
                elif risk[i] == risk[j]: num += 0.5
                cnt += 1
                if cnt > max_pairs: break
        if cnt > max_pairs: break
    return num / max(den, 1)

haz_prob = 1 / (1 + np.exp(-test_out['hazard']))
cum_risk = haz_prob.sum(axis=1)
c_idx = c_index(test_out['event_time'], test_out['event_ind'], cum_risk)

results_07 = dict(
    UPDRS_all=metr_all, UPDRS_PD=metr_pd, UPDRS_Prodromal=metr_pro,
    NSD_ISS_weighted_F1=float(stage_f1_w),
    NSD_ISS_macro_F1=float(stage_f1_m),
    DaTscan_Pearson_r=float(r_dat),  DaTscan_Pearson_p=float(p_dat),
    DaTscan_Spearman_r=float(rs_dat), DaTscan_Spearman_p=float(ps_dat),
    NHH_c_index=float(c_idx),
)
with open(RESULTS / 'block_07_metrics.json', 'w') as f:
    json.dump(results_07, f, indent=2, default=str)

print('\n=== Block 07 primary metrics ===')
for k, v in results_07.items(): print(f'  {k}: {v}')


## Block 08 — Uncertainty Calibration & Clinical Monitoring

ECE across deciles of predictive std, MAE per epistemic-uncertainty quintile, uncertainty ratio, and the **clinical monitoring odds ratio** — top-Q vs bottom-Q u_epi, outcome = |ΔUPDRS@36mo| > 15 — with 95% CI and χ² p-value.

In [ ]:

# ==== Block 08 — uncertainty calibration + monitoring signal ====
from scipy.stats import chi2_contingency
import math

u_epi = test_out['beta'] / (test_out['nu'] * np.clip(test_out['alpha'] - 1, 1e-6, None))
u_ale = test_out['beta'] / np.clip(test_out['alpha'] - 1, 1e-6, None)
obs_m = test_out['y_obs'].astype(bool)

y_abs = np.abs(test_out['y_u3'] * CFG['UPDRS_MAX'] - test_out['gamma'] * CFG['UPDRS_MAX'])

# ECE over deciles of predicted std
pred_std = np.sqrt(u_epi + u_ale) * CFG['UPDRS_MAX']
bins = np.quantile(pred_std[obs_m], np.linspace(0, 1, 11))
ece = 0.0
for i in range(10):
    m = obs_m & (pred_std >= bins[i]) & (pred_std < bins[i+1] + (1e-9 if i==9 else 0))
    if m.sum() < 5: continue
    frac     = m.sum() / obs_m.sum()
    expected = pred_std[m].mean()
    actual   = y_abs[m].mean()
    ece += frac * abs(expected - actual) / CFG['UPDRS_MAX']

# MAE per quintile of epistemic uncertainty
qs = np.quantile(u_epi[obs_m], [0.2, 0.4, 0.6, 0.8])
mae_quintiles = []
for lo, hi in zip([-np.inf] + list(qs), list(qs) + [np.inf]):
    m = obs_m & (u_epi >= lo) & (u_epi < hi)
    mae_quintiles.append(float(y_abs[m].mean()) if m.sum() >= 5 else float('nan'))
unc_ratio = mae_quintiles[-1] / max(mae_quintiles[0], 1e-6)

# Clinical monitoring signal: baseline vs V06 UPDRS for top-Q vs bot-Q u_epi
bl_map  = dict(zip(master_df['PATNO'].values, master_df['BL_UPDRS3'].values))
v06_map = dict(zip(master_df['PATNO'].values, master_df['V06_UPDRS3'].values))
BL  = np.array([bl_map.get(int(p), np.nan)  for p in test_out['patno']])
V6  = np.array([v06_map.get(int(p), np.nan) for p in test_out['patno']])
dlt = np.abs(V6 - BL)
dlt_obs = ~np.isnan(dlt)
big_change = (dlt > 15).astype(int)

q75, q25 = np.quantile(u_epi[obs_m], [0.75, 0.25])
top_q = (u_epi >= q75) & obs_m & dlt_obs
bot_q = (u_epi <= q25) & obs_m & dlt_obs
a = int(((top_q) & (big_change == 1)).sum())
b = int(((top_q) & (big_change == 0)).sum())
c = int(((bot_q) & (big_change == 1)).sum())
d = int(((bot_q) & (big_change == 0)).sum())
or_val = (a * d) / max(b * c, 1e-6)
log_or = math.log(max(or_val, 1e-6))
se     = math.sqrt(1/max(a,1) + 1/max(b,1) + 1/max(c,1) + 1/max(d,1))
ci_low, ci_hi = math.exp(log_or - 1.96*se), math.exp(log_or + 1.96*se)

table = np.array([[a, b], [c, d]])
if table.sum() > 0 and (table.min() >= 0) and table.all():
    chi2, p_chi, _, _ = chi2_contingency(table)
else:
    chi2, p_chi = float('nan'), float('nan')

calibration = dict(
    ECE=float(ece),
    MAE_quintiles=mae_quintiles,
    uncertainty_ratio=float(unc_ratio),
    clinical_table=dict(a=a, b=b, c=c, d=d),
    clinical_OR=float(or_val),
    clinical_OR_CI=[float(ci_low), float(ci_hi)],
    chi2=float(chi2), chi2_p=float(p_chi),
)
with open(RESULTS / 'block_08_calibration.json', 'w') as f:
    json.dump(calibration, f, indent=2, default=str)

print(f'ECE: {ece:.4f}')
print(f'MAE quintiles: {[round(x,2) for x in mae_quintiles]}')
print(f'Uncertainty ratio (Q5/Q1): {unc_ratio:.2f}')
print(f'Clinical monitoring OR: {or_val:.2f}  95% CI [{ci_low:.2f}, {ci_hi:.2f}]  chi2 p={p_chi:.4f}')
print('>> Clinical implication: patients in the top quartile of baseline epistemic uncertainty are ',
      f'~{or_val:.1f}x more likely to experience UPDRS change > 15 points at 36 months — ',
      'justifying 6-month monitoring rather than 12-month for this subgroup.')


## Block 09 — Ablation Study (B0 through B5)

- **B0** XGBoost + full enriched features (incl. LTP)
- **B1** Temporal transformer only (no graph, no GCN, no LTP)
- **B2** + Fixed A_braak (shared graph)
- **B3** + PSBGL deterministic (patient-specific, no stochastic, no GCN)
- **B4** + GMP-PBG (PSBGL deterministic + graph convolution)
- **B5** Full v2.1 with S-PBGL (use the model already trained in Block 06)

Ablation variants B1–B4 are trained with a reduced schedule (40 epochs) so the whole sweep is tractable on T4; bump `ABLATION_EPOCHS` for paper-grade results.

In [ ]:

# ==== Block 09 — ablation study ====
BLOCK_09_CKPT = RESULTS / 'block_09_ablation.json'
ABLATION_EPOCHS = 40   # bump to 150 for paper-grade results

# ---- B0: XGBoost + LTP + all static features ----
from xgboost import XGBRegressor
X_all = np.concatenate([_data['X_clin'], _data['X_gen'], _data['X_img'], _data['X_bio'], _data['X_ltp']], axis=1)
y_all = (_data['target_u3_norm'] * CFG['UPDRS_MAX'])
obs_all = _data['target_observed'].astype(bool)

def r2_subset(y_true, y_pred, idx_mask):
    m = idx_mask.astype(bool)
    if m.sum() < 10: return float('nan')
    return float(r2_score(y_true[m], y_pred[m]))

tr_obs_mask = obs_all.copy(); tr_obs_mask[~np.isin(np.arange(len(y_all)), idx_tr)] = False
xgb = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=SEED, n_jobs=4,
                   tree_method='hist')
xgb.fit(X_all[idx_tr][obs_all[idx_tr]], y_all[idx_tr][obs_all[idx_tr]])
pred_te = xgb.predict(X_all[idx_te])
obs_te_v = obs_all[idx_te]
pd_mask_te = (master_df.loc[idx_te, 'cohort'].values == 'PD')
r2_b0_all = r2_score(y_all[idx_te][obs_te_v], pred_te[obs_te_v])
r2_b0_pd  = r2_subset(y_all[idx_te], pred_te, obs_te_v & pd_mask_te)
xgb_feat_imp = xgb.feature_importances_
print(f'B0 XGBoost: R2_all={r2_b0_all:.4f}  R2_PD={r2_b0_pd:.4f}')

# ---- B1..B4: transformer variants with feature-flag gating ----
class BrainFormerAblation(nn.Module):
    '''Configurable variant. `use_graph` ∈ {None,'fixed','psbgl','spbgl'}; `use_gcn`, `use_ltp` bool.'''
    def __init__(self, d_clin, d_gen, d_img, d_bio, d_ltp, A_braak,
                 use_graph='spbgl', use_gcn=True, use_ltp=True, cfg=CFG):
        super().__init__()
        self.use_graph, self.use_gcn, self.use_ltp = use_graph, use_gcn, use_ltp
        self.mod_enc = MultiModalEncoder(d_clin, d_gen, d_img, d_bio, cfg['d_model'])
        if use_ltp:
            self.ltp_enc = LTPEncoder(d_ltp, cfg['d_model'])
        if use_graph == 'spbgl':
            self.graph = StochasticBraakGraph(A_braak, cfg['d_model'], cfg['n_nodes'], cfg['d_node'])
        elif use_graph == 'psbgl':
            class PSBGLDet(nn.Module):  # deterministic personalised graph
                def __init__(self, A_braak, d_model=128, n_nodes=10, d_node=8, eps=0.05):
                    super().__init__()
                    self.register_buffer('A_braak', A_braak.float())
                    self.eps = eps; self.n_nodes = n_nodes; self.d_node = d_node
                    self.node_proj = nn.Sequential(nn.Linear(d_model, n_nodes*d_node), nn.Tanh())
                    self.scale = nn.Parameter(torch.ones(1)*0.1)
                def forward(self, z, node_miss=None):
                    B = z.shape[0]
                    F_i = self.node_proj(z).reshape(B, self.n_nodes, self.d_node)
                    S   = torch.bmm(F_i, F_i.transpose(1,2)) / (self.d_node**0.5)
                    A_p = (self.A_braak + self.eps).unsqueeze(0)
                    A_r = torch.sigmoid(self.scale * S) * A_p
                    d   = A_r.sum(-1, keepdim=True).clamp(min=1e-6)
                    d_is= d.pow(-0.5)
                    A_i = d_is * A_r * d_is.transpose(1,2)
                    return A_i, torch.zeros_like(A_i), torch.tensor(0.0, device=z.device)
                def graph_reg_loss(self, A_i):
                    return F.mse_loss(A_i, self.A_braak.unsqueeze(0).expand_as(A_i))
            self.graph = PSBGLDet(A_braak, cfg['d_model'], cfg['n_nodes'], cfg['d_node'])
        # fixed / None handled inline
        self.register_buffer('A_fixed', A_braak.float())
        if use_gcn and use_graph is not None:
            self.gcn = BraakGraphConv(d_img, cfg['d_model'], cfg['n_nodes'])
        self.temporal = BSMTA(cfg['d_model'], D_VISIT, cfg['n_heads'], cfg['n_nodes'],
                              cfg['T_MAX'], cfg['n_tf_layers'])
        self.h_edp = EDPHead(cfg['d_model'])
        self.h_stg = StageHead(cfg['d_model'], max(N_STAGES, 7))
        self.h_sbr = DaTscanHead(cfg['d_model'])
        self.h_nhh = NeuralHazardHead(cfg['d_model'], cfg['T_MAX'])

    def forward(self, batch):
        z = self.mod_enc(batch['clin'], batch['gen'], batch['img'], batch['bio'])
        if self.use_ltp:
            z = z + self.ltp_enc(batch['ltp'])
        kl = torch.tensor(0.0, device=z.device)
        if self.use_graph == 'fixed':
            A_i = self.A_fixed.unsqueeze(0).expand(z.shape[0], -1, -1)
            d = A_i.sum(-1, keepdim=True).clamp(min=1e-6)
            d_is = d.pow(-0.5); A_i = d_is * A_i * d_is.transpose(1,2)
        elif self.use_graph in ('psbgl','spbgl'):
            A_i, _, kl = self.graph(z, batch.get('node_miss'))
        else:
            A_i = torch.eye(CFG['n_nodes'], device=z.device).unsqueeze(0).expand(z.shape[0], -1, -1)
        if self.use_gcn and self.use_graph is not None:
            z = z + self.gcn(batch['img_raw'], A_i)
        h, _ = self.temporal(batch['visit'], A_i, z_ctx=z, visit_mask=batch.get('visit_mask'))
        gamma, nu, alpha, beta = self.h_edp(h)
        return dict(gamma=gamma, nu=nu, alpha=alpha, beta=beta,
                    stage_logits=self.h_stg(h), sbr_pred=self.h_sbr(h),
                    hazard_logits=self.h_nhh(h), A_i=A_i, kl_graph=kl)

def train_variant(tag, use_graph, use_gcn, use_ltp, epochs=ABLATION_EPOCHS):
    set_seed()
    m = BrainFormerAblation(D_CLIN, D_GEN, D_IMG, D_BIO, D_LTP, A_braak_t,
                            use_graph=use_graph, use_gcn=use_gcn, use_ltp=use_ltp).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    for e in range(epochs):
        m.train()
        for batch in loader_tr:
            b = move(batch, DEVICE)
            out = m(b)
            y, y_obs = b['y_updrs'], b['y_obs']
            two_b = 2.0 * out['beta'] * (1 + out['nu'])
            nll = 0.5*torch.log(math.pi/out['nu']) - out['alpha']*torch.log(two_b) \
                + (out['alpha']+0.5)*torch.log(out['nu']*(y-out['gamma'])**2 + two_b) \
                + torch.lgamma(out['alpha']) - torch.lgamma(out['alpha']+0.5)
            reg = torch.abs(y - out['gamma']) * (2*out['nu'] + out['alpha'])
            L = ((nll + 0.01*reg) * y_obs).sum() / y_obs.sum().clamp(min=1)
            L = L + CFG['w_stage'] * F.cross_entropy(out['stage_logits'], b['y_stage'], weight=class_weights)
            opt.zero_grad(); L.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), CFG['grad_clip']); opt.step()
    # test R2
    m.eval()
    ys, ps, obs = [], [], []
    cs = []
    with torch.no_grad():
        for batch in loader_te:
            b = move(batch, DEVICE); out = m(b)
            ys.append(b['y_updrs'].cpu().numpy()); ps.append(out['gamma'].cpu().numpy())
            obs.append(b['y_obs'].cpu().numpy()); cs.append(b['patno'].cpu().numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps); obs = np.concatenate(obs)
    patnos = np.concatenate(cs)
    cohort_te = np.array([cohort_by_patno.get(int(p),'?') for p in patnos])
    mm = obs.astype(bool)
    r2_all = float(r2_score(ys[mm]*CFG['UPDRS_MAX'], ps[mm]*CFG['UPDRS_MAX']))
    pd_m = mm & (cohort_te == 'PD')
    r2_pd = float(r2_score(ys[pd_m]*CFG['UPDRS_MAX'], ps[pd_m]*CFG['UPDRS_MAX'])) if pd_m.sum()>=10 else float('nan')
    print(f'{tag}: R2_all={r2_all:.4f}  R2_PD={r2_pd:.4f}')
    return dict(R2_all=r2_all, R2_PD=r2_pd)

abl = {}
if BLOCK_09_CKPT.exists():
    abl = json.loads(BLOCK_09_CKPT.read_text())
    print('Resumed ablation results.')
else:
    abl['B0'] = dict(R2_all=float(r2_b0_all), R2_PD=float(r2_b0_pd), name='XGBoost+LTP')
    abl['B1'] = {**train_variant('B1', None,     False, False), 'name':'Transformer only'}
    abl['B2'] = {**train_variant('B2', 'fixed',  True,  False), 'name':'+ fixed Braak'}
    abl['B3'] = {**train_variant('B3', 'psbgl',  False, False), 'name':'+ PSBGL det'}
    abl['B4'] = {**train_variant('B4', 'psbgl',  True,  True),  'name':'+ GMP-PBG'}
    # B5 — use already-trained S-PBGL full model (from Block 06)
    mm = test_out['y_obs'].astype(bool)
    r2_b5_all = float(r2_score(test_out['y_u3'][mm]*CFG['UPDRS_MAX'], test_out['gamma'][mm]*CFG['UPDRS_MAX']))
    pd_mm = mm & pd_mask
    r2_b5_pd  = float(r2_score(test_out['y_u3'][pd_mm]*CFG['UPDRS_MAX'], test_out['gamma'][pd_mm]*CFG['UPDRS_MAX'])) if pd_mm.sum()>=10 else float('nan')
    abl['B5'] = dict(R2_all=r2_b5_all, R2_PD=r2_b5_pd, name='Full v2.1 (S-PBGL)')
    with open(BLOCK_09_CKPT, 'w') as f: json.dump(abl, f, indent=2)

print('\n=== Ablation (primary R² on PD cohort) ===')
for k in sorted(abl.keys()):
    v = abl[k]; print(f'  {k} {v["name"]:25s}  R²_all={v["R2_all"]:.4f}  R²_PD={v["R2_PD"]:.4f}')


## Block 10 — Statistical Tests

Wilcoxon signed-rank on absolute errors (BrainFormer vs XGBoost), 1000-iter bootstrap 95% CI on ΔR² and DaTscan Pearson r.

In [ ]:

# ==== Block 10 — Wilcoxon + bootstrap CI ====
from scipy.stats import wilcoxon

obs_te_v = _data['target_observed'][idx_te].astype(bool)
y_te = (_data['target_u3_norm'][idx_te] * CFG['UPDRS_MAX'])
p_bf = test_out['gamma'] * CFG['UPDRS_MAX']
m = obs_te_v & test_out['y_obs'].astype(bool)

p_xgb_full = xgb.predict(X_all[idx_te])

err_bf  = np.abs(y_te[m] - p_bf[m])
err_xgb = np.abs(y_te[m] - p_xgb_full[m])
stat, p_wil = wilcoxon(err_bf, err_xgb)

def bootstrap_r2_delta(n_boot=1000):
    rng = np.random.default_rng(SEED); dr2 = []
    for _ in range(n_boot):
        idx = rng.choice(m.sum(), size=m.sum(), replace=True)
        r2b = r2_score(y_te[m][idx], p_bf[m][idx])
        r2x = r2_score(y_te[m][idx], p_xgb_full[m][idx])
        dr2.append(r2b - r2x)
    return np.percentile(dr2, [2.5, 50, 97.5])

def bootstrap_pearson(n_boot=1000):
    sbr_obs = test_out['sbr_obs'].astype(bool)
    if sbr_obs.sum() < 30:
        return np.array([float('nan')]*3)
    x = test_out['y_sbr'][sbr_obs]; y = test_out['sbr'][sbr_obs]
    rng = np.random.default_rng(SEED); rs = []
    for _ in range(n_boot):
        idx = rng.choice(len(x), size=len(x), replace=True)
        rs.append(pearsonr(x[idx], y[idx])[0])
    return np.percentile(rs, [2.5, 50, 97.5])

ci_dr2 = bootstrap_r2_delta()
ci_rdat= bootstrap_pearson()

stats = dict(
    wilcoxon_W=float(stat), wilcoxon_p=float(p_wil),
    delta_R2_ci=[float(x) for x in ci_dr2],
    DaTscan_r_ci=[float(x) for x in ci_rdat],
)
with open(RESULTS/'block_10_stats.json','w') as f:
    json.dump(stats, f, indent=2, default=str)

print(f'Wilcoxon W={stat:.1f}  p={p_wil:.4e}')
print(f'ΔR² (BF − XGB) 95% CI: [{ci_dr2[0]:+.4f}, {ci_dr2[2]:+.4f}]  median={ci_dr2[1]:+.4f}')
print(f'DaTscan Pearson r 95% CI: [{ci_rdat[0]:.4f}, {ci_rdat[2]:.4f}]  median={ci_rdat[1]:.4f}')


## Block 11 — GBA-Stratified Analysis

In [ ]:

# ==== Block 11 — GBA stratified ====
from scipy.stats import mannwhitneyu

gba_pos = test_out['GBA'].astype(bool)
mm = test_out['y_obs'].astype(bool)

def group_metrics(mask):
    if mask.sum() < 10: return None
    y = test_out['y_u3'][mask] * CFG['UPDRS_MAX']
    p = test_out['gamma'][mask] * CFG['UPDRS_MAX']
    return dict(R2=float(r2_score(y, p)), MAE=float(mean_absolute_error(y, p)), n=int(mask.sum()))

mpos = mm & gba_pos; mneg = mm & ~gba_pos
gba_results = dict(
    GBA_pos=group_metrics(mpos),
    GBA_neg=group_metrics(mneg),
)
# Mann-Whitney U on epistemic uncertainty
if mpos.sum() >= 5 and mneg.sum() >= 5:
    stat, p_mw = mannwhitneyu(u_epi[mpos], u_epi[mneg], alternative='two-sided')
    gba_results['u_epi_MW'] = dict(U=float(stat), p=float(p_mw),
                                   mean_pos=float(u_epi[mpos].mean()),
                                   mean_neg=float(u_epi[mneg].mean()))
# Bootstrap ΔR² GBA+ vs GBA-
def bootstrap_delta_r2_groups(n_boot=500):
    rng = np.random.default_rng(SEED); d = []
    yp = test_out['y_u3'][mpos]*CFG['UPDRS_MAX']; pp = test_out['gamma'][mpos]*CFG['UPDRS_MAX']
    yn = test_out['y_u3'][mneg]*CFG['UPDRS_MAX']; pn = test_out['gamma'][mneg]*CFG['UPDRS_MAX']
    for _ in range(n_boot):
        ip = rng.choice(len(yp), size=len(yp), replace=True)
        ine= rng.choice(len(yn), size=len(yn), replace=True)
        d.append(r2_score(yp[ip], pp[ip]) - r2_score(yn[ine], pn[ine]))
    return np.percentile(d, [2.5, 50, 97.5])

if mpos.sum() >= 30 and mneg.sum() >= 30:
    ci = bootstrap_delta_r2_groups()
    gba_results['delta_R2_ci'] = [float(x) for x in ci]

with open(RESULTS/'block_11_gba.json','w') as f:
    json.dump(gba_results, f, indent=2, default=str)

print('\n=== GBA-stratified ===')
for k,v in gba_results.items(): print(f'  {k}: {v}')


## Block 12 — SAA-Stratified Analysis *(NEW)*

Mirrors Block 11 for SAA status, **plus** a Mann-Whitney test on `A_i` edge weights at the SNc (node 2) and LC (node 1) — the testable claim is that SAA+ patients have stronger early-Braak connectivity.

In [ ]:

# ==== Block 12 — SAA stratified ====
saa_pos = test_out['SAA'].astype(bool)
mm = test_out['y_obs'].astype(bool)
mpos_s = mm & saa_pos; mneg_s = mm & ~saa_pos

saa_results = dict(
    SAA_pos=group_metrics(mpos_s),
    SAA_neg=group_metrics(mneg_s),
)
if mpos_s.sum() >= 5 and mneg_s.sum() >= 5:
    stat, p_mw = mannwhitneyu(u_epi[mpos_s], u_epi[mneg_s], alternative='two-sided')
    saa_results['u_epi_MW'] = dict(U=float(stat), p=float(p_mw))

A_te = test_out['A']                           # [N_test, 10, 10]
snc_row = A_te[:, 2, :].mean(axis=1)            # mean SNc edge strength per patient
lc_row  = A_te[:, 1, :].mean(axis=1)
if saa_pos.sum() >= 10 and (~saa_pos).sum() >= 10:
    u_snc, p_snc = mannwhitneyu(snc_row[saa_pos], snc_row[~saa_pos], alternative='two-sided')
    u_lc,  p_lc  = mannwhitneyu(lc_row[saa_pos],  lc_row[~saa_pos],  alternative='two-sided')
    saa_results['edge_MW_SNc'] = dict(U=float(u_snc), p=float(p_snc),
                                      mean_pos=float(snc_row[saa_pos].mean()),
                                      mean_neg=float(snc_row[~saa_pos].mean()))
    saa_results['edge_MW_LC']  = dict(U=float(u_lc),  p=float(p_lc),
                                      mean_pos=float(lc_row[saa_pos].mean()),
                                      mean_neg=float(lc_row[~saa_pos].mean()))

with open(RESULTS/'block_12_saa.json','w') as f:
    json.dump(saa_results, f, indent=2, default=str)

print('\n=== SAA-stratified ===')
for k,v in saa_results.items(): print(f'  {k}: {v}')


## Block 13 — Graph Topology Phenotyping *(NEW)*

Flatten each test patient's `A_i` upper triangle (45 edges), KMeans(k=3), then label clusters based on which node group (early brainstem / limbic / neocortical) carries the largest mean edge weight. Fisher's exact test for GBA+ enrichment per cluster.

In [ ]:

# ==== Block 13 — graph topology phenotyping ====
from sklearn.cluster import KMeans
from scipy.stats import fisher_exact
set_seed()

triu = np.triu_indices(CFG['n_nodes'], k=1)                   # 45 edges
A_vec = test_out['A'][:, triu[0], triu[1]]                    # [N_test, 45]
km = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(A_vec)
cluster = km.labels_                                          # [N_test]

# Node groupings for cluster labelling
brainstem = [0, 1, 2]        # DMV, LC, SNc
limbic    = [3, 4, 5]        # Amygdala, Hippocampus, Entorhinal
cortical  = [6, 7, 8, 9]     # Temporal, Associative, Motor, Prefrontal

def cluster_label(mean_A):
    score = dict(
        brainstem=mean_A[np.ix_(brainstem, brainstem)].mean(),
        limbic=mean_A[np.ix_(limbic, limbic)].mean(),
        cortical=mean_A[np.ix_(cortical, cortical)].mean(),
    )
    return max(score, key=score.get), score

cluster_info = {}
gba_te = test_out['GBA'].astype(bool)
for cid in range(3):
    mask = (cluster == cid)
    mean_A = test_out['A'][mask].mean(axis=0)
    lab, sc = cluster_label(mean_A)
    # Fisher exact: GBA+ in this cluster vs rest
    a = int((mask & gba_te).sum()); b = int((mask & ~gba_te).sum())
    c = int((~mask & gba_te).sum()); d = int((~mask & ~gba_te).sum())
    if min(a+b, c+d) > 0 and min(a+c, b+d) > 0:
        odds, p = fisher_exact([[a, b], [c, d]])
    else:
        odds, p = float('nan'), float('nan')
    cluster_info[f'cluster_{cid}'] = dict(
        label=lab, score=sc, n=int(mask.sum()),
        gba_pos=a, gba_neg=b, fisher_odds=float(odds), fisher_p=float(p),
        mean_A=mean_A.tolist()
    )
np.save(RESULTS/'block_13_cluster_ids.npy', cluster)
with open(RESULTS/'block_13_topology.json','w') as f:
    json.dump(cluster_info, f, indent=2, default=str)

print('\n=== Graph topology phenotypes ===')
for k, v in cluster_info.items():
    print(f"  {k}: label={v['label']} n={v['n']} GBA+%={100*v['gba_pos']/max(v['n'],1):.1f}  Fisher_p={v['fisher_p']:.4f}")


## Block 14 — Two-Source Uncertainty Analysis *(NEW)*

Decompose total uncertainty into `u_pred` (NIG epistemic, from outcome head) and `u_graph` (mean of `u_edge`, from S-PBGL posterior). Pearson correlation + identification of the **"biologically-atypical-but-clinically-predictable" phenotype** (top quartile u_graph ∩ bottom quartile u_pred).

In [ ]:

# ==== Block 14 — two-source uncertainty ====
u_graph = test_out['u_edge'].mean(axis=(1, 2))            # [N_test]
u_pred  = u_epi

r_ug, p_ug = pearsonr(u_graph, u_pred)

q_g_hi = np.quantile(u_graph, 0.75)
q_p_lo = np.quantile(u_pred,  0.25)
phenotype_mask = (u_graph >= q_g_hi) & (u_pred <= q_p_lo)

gba_pheno = int((phenotype_mask & gba_pos).sum())
gba_other = int((~phenotype_mask & gba_pos).sum())
other_all = int((~phenotype_mask).sum())
enrichment = (gba_pheno / max(phenotype_mask.sum(), 1)) / (gba_other / max(other_all, 1) + 1e-9)

updrs_pheno = test_out['y_u3'][phenotype_mask] * CFG['UPDRS_MAX']
updrs_rest  = test_out['y_u3'][~phenotype_mask] * CFG['UPDRS_MAX']

two_source = dict(
    pearson_r_ugraph_upred=float(r_ug),
    pearson_p_ugraph_upred=float(p_ug),
    phenotype_n=int(phenotype_mask.sum()),
    phenotype_gba_fraction=float(gba_pheno/max(phenotype_mask.sum(),1)),
    phenotype_gba_enrichment=float(enrichment),
    phenotype_updrs_mean=float(updrs_pheno.mean()) if phenotype_mask.sum()>0 else float('nan'),
    phenotype_updrs_std=float(updrs_pheno.std()) if phenotype_mask.sum()>0 else float('nan'),
    rest_updrs_mean=float(updrs_rest.mean()),
)
np.save(RESULTS/'block_14_u_graph.npy', u_graph)
np.save(RESULTS/'block_14_u_pred.npy',  u_pred)
with open(RESULTS/'block_14_two_source.json','w') as f:
    json.dump(two_source, f, indent=2, default=str)

print('\n=== Two-source uncertainty ===')
for k,v in two_source.items(): print(f'  {k}: {v}')


## Block 15 — Subtype Clustering on GBA-BCSD Latent Space

KMeans(k=4) on the L2-normed projection head output of the test set. Silhouette, GBA distribution per cluster, and per-cluster mean epistemic uncertainty + mean LTP c1 velocity (UPDRS-III).

In [ ]:

# ==== Block 15 — latent subtype clustering ====
from sklearn.metrics import silhouette_score
set_seed()

Z = test_out['proj']              # [N_test, 64] unit-normed
km4 = KMeans(n_clusters=4, n_init=20, random_state=SEED).fit(Z)
sil = silhouette_score(Z, km4.labels_)
print(f'Silhouette (k=4): {sil:.4f}')

ltp_te = _data['X_ltp'][idx_te]
c1_u3  = ltp_te[:, 1]             # c1 for UPDRS3 (velocity)

sub = {}
for cid in range(4):
    mask = km4.labels_ == cid
    sub[f'cluster_{cid}'] = dict(
        n=int(mask.sum()),
        gba_frac=float(test_out['GBA'][mask].mean()) if mask.sum()>0 else float('nan'),
        saa_frac=float(test_out['SAA'][mask].mean()) if mask.sum()>0 else float('nan'),
        u_epi_mean=float(u_epi[mask].mean()) if mask.sum()>0 else float('nan'),
        c1_velocity_mean=float(c1_u3[mask].mean()) if mask.sum()>0 else float('nan'),
        updrs_mean=float(test_out['y_u3'][mask].mean()*CFG['UPDRS_MAX']) if mask.sum()>0 else float('nan'),
    )

subtyping = dict(silhouette=float(sil), clusters=sub)
np.save(RESULTS/'block_15_cluster_ids.npy', km4.labels_)
with open(RESULTS/'block_15_subtyping.json','w') as f:
    json.dump(subtyping, f, indent=2, default=str)

print('\n=== Latent subtypes ===')
for k,v in sub.items():
    print(f'  {k}: n={v["n"]}  GBA+%={v["gba_frac"]*100:.1f}  SAA+%={v["saa_frac"]*100:.1f}  u_epi={v["u_epi_mean"]:.4f}  c1_UPDRS3_vel={v["c1_velocity_mean"]:.3f}')


## Block 16 — Figures (12 original + 4 new)

All figures are written to `FIGURES/` as both `.pdf` (paper) and `.png` (inspection).

In [ ]:

# ==== Block 16 — figures ====
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200,
                     'font.size': 10, 'axes.titlesize': 12})

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIGURES / f'{name}.png', bbox_inches='tight')
    plt.close(fig)
    print('saved', name)

# Fig 1 — cohort overview
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
master_df['cohort'].value_counts().plot.bar(ax=axes[0], color='steelblue'); axes[0].set_title('Cohort')
sns.histplot(master_df['V06_UPDRS3'].dropna(), ax=axes[1], bins=30, color='teal'); axes[1].set_title('V06 UPDRS-III distribution')
sns.countplot(x='NSD_ISS_stage', data=master_df, ax=axes[2], color='darkorange'); axes[2].set_title('NSD-ISS stage distribution')
save_fig(fig, 'fig01_cohort')

# Fig 2 — canonical Braak adjacency
fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(A_braak, xticklabels=NODE_NAMES, yticklabels=NODE_NAMES, cmap='Blues', ax=ax, cbar_kws={'label':'edge'})
ax.set_title('Canonical Braak adjacency (A_braak)')
save_fig(fig, 'fig02_Abraak')

# Fig 3 — UPDRS scatter coloured by u_epi
fig, ax = plt.subplots(figsize=(5.5, 5))
mm = test_out['y_obs'].astype(bool)
sc = ax.scatter(test_out['y_u3'][mm]*CFG['UPDRS_MAX'], test_out['gamma'][mm]*CFG['UPDRS_MAX'],
                c=u_epi[mm], cmap='viridis', s=10, alpha=0.6)
lims = [0, CFG['UPDRS_MAX']]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('True UPDRS-III @ 36mo'); ax.set_ylabel('Predicted γ')
plt.colorbar(sc, ax=ax, label='epistemic u'); ax.set_title('Fig 3 — UPDRS scatter')
save_fig(fig, 'fig03_scatter_updrs')

# Fig 4 — calibration (quintiles)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(5), mae_quintiles, color='firebrick'); ax.set_xticks(range(5))
ax.set_xticklabels(['Q1','Q2','Q3','Q4','Q5']); ax.set_ylabel('MAE (UPDRS units)')
ax.set_title(f'MAE by epistemic quintile (ratio={unc_ratio:.2f})')
save_fig(fig, 'fig04_calibration')

# Fig 5 — latent PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit_transform(test_out['z'])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for cid in range(4):
    m = km4.labels_ == cid
    axes[0].scatter(pca[m, 0], pca[m, 1], s=8, label=f'c{cid}', alpha=0.6)
axes[0].legend(); axes[0].set_title('PCA by cluster')
axes[1].scatter(pca[~gba_pos, 0], pca[~gba_pos, 1], s=8, color='#888', alpha=0.4, label='GBA-')
axes[1].scatter(pca[ gba_pos, 0], pca[ gba_pos, 1], s=14, color='crimson', alpha=0.8, label='GBA+')
axes[1].legend(); axes[1].set_title('PCA by GBA status')
save_fig(fig, 'fig05_pca_latent')

# Fig 6 — NSD-ISS confusion matrix
fig, ax = plt.subplots(figsize=(5, 4.5))
n_stg = max(N_STAGES, 7)
cm = np.zeros((n_stg, n_stg), dtype=int)
for t, p in zip(test_out['y_stage'], stage_pred): cm[int(t), int(p)] += 1
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax)
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title(f'NSD-ISS confusion  W-F1={stage_f1_w:.3f}')
save_fig(fig, 'fig06_nsdiss_cm')

# Fig 7 — GBA-stratified scatter + uncertainty box
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
yp = test_out['y_u3']*CFG['UPDRS_MAX']; pp = test_out['gamma']*CFG['UPDRS_MAX']
axes[0].scatter(yp[mm & ~gba_pos], pp[mm & ~gba_pos], s=10, c='#888', label='GBA-', alpha=0.5)
axes[0].scatter(yp[mm &  gba_pos], pp[mm &  gba_pos], s=18, c='crimson', label='GBA+', alpha=0.8)
axes[0].plot([0, 132], [0, 132], 'k--', lw=1); axes[0].legend(); axes[0].set_title('UPDRS by GBA status')
axes[0].set_xlabel('true'); axes[0].set_ylabel('pred')
data_box = [u_epi[~gba_pos & mm], u_epi[gba_pos & mm]]
axes[1].boxplot(data_box, tick_labels=['GBA-', 'GBA+'] if hasattr(axes[1], 'set_xticklabels') else None)
axes[1].set_title('Epistemic uncertainty by GBA'); axes[1].set_ylabel('u_epi')
save_fig(fig, 'fig07_gba_stratified')

# Fig 8 — ablation bar chart
fig, ax = plt.subplots(figsize=(8, 4))
order = ['B0','B1','B2','B3','B4','B5']
r2s   = [abl[k]['R2_PD'] for k in order]
names = [abl[k]['name']  for k in order]
bars = ax.bar(order, r2s, color=['#bbb','#aab','#99b','#88b','#66a','#264'])
ax.set_ylabel('R² (PD cohort)'); ax.set_title('Ablation')
for b, n, r in zip(bars, names, r2s):
    ax.text(b.get_x()+b.get_width()/2, r+0.005, f'{r:.3f}', ha='center', fontsize=8)
    ax.text(b.get_x()+b.get_width()/2, -0.05, n, ha='center', fontsize=7, rotation=20)
save_fig(fig, 'fig08_ablation')

# Fig 9 — BSMTA per-head temporal-node queries
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
Q = model.temporal.layers[0].temporal_node_queries.detach().cpu().numpy()   # [H, T, n_nodes]
for h in range(Q.shape[0]):
    ax = axes[h//4, h%4]
    sns.heatmap(Q[h], yticklabels=['BL','V02','V04','V06'][:Q.shape[1]],
                xticklabels=NODE_NAMES, ax=ax, cmap='coolwarm', center=0, cbar=False)
    ax.set_title(f'head {h}')
fig.suptitle('Fig 9 — BSMTA temporal-node query heatmaps')
save_fig(fig, 'fig09_bsmta_heads')

# Fig 10 — XGB feature importance (top 20)
feat_names = (CLINICAL_FEATURES + GENETIC_FEATURES + IMAGING_FEATURES + BIOSPECIMEN_FEATURES
              + [f'ltp_{i}' for i in range(_data['X_ltp'].shape[1])])
n_cm = X_clin.shape[1] - len(CLINICAL_FEATURES)
# pad missingness flags
feat_names_full = []
for grp, names in [('clin', CLINICAL_FEATURES), ('gen', GENETIC_FEATURES),
                   ('img', IMAGING_FEATURES), ('bio', BIOSPECIMEN_FEATURES)]:
    feat_names_full += [str(n) for n in names] + [f'{grp}_miss_{i}' for i in range(len(names))]
feat_names_full += [f'ltp_{i}' for i in range(_data['X_ltp'].shape[1])]
# trim or pad to match xgb_feat_imp
fn = (feat_names_full + ['?']*max(0, len(xgb_feat_imp)-len(feat_names_full)))[:len(xgb_feat_imp)]
order20 = np.argsort(xgb_feat_imp)[::-1][:20]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(range(20)[::-1], xgb_feat_imp[order20], color='teal')
ax.set_yticks(range(20)[::-1]); ax.set_yticklabels([fn[i] for i in order20], fontsize=8)
ax.set_title('Fig 10 — XGBoost top-20 features')
save_fig(fig, 'fig10_xgb_importance')

# Fig 11 — survival curves by GBA × baseline severity
fig, ax = plt.subplots(figsize=(6, 4.5))
S_te = 1 / (1 + np.exp(-test_out['hazard']))
S_cum = np.cumprod(1 - S_te, axis=1)
t_axis = np.array([0, 12, 24, 36])[:S_cum.shape[1]]
bl = np.array([bl_map.get(int(p), np.nan) for p in test_out['patno']])
bl_hi = bl > np.nanmedian(bl)
for lab, m in [('GBA- / BL low',  ~gba_pos & ~bl_hi),
               ('GBA- / BL high', ~gba_pos &  bl_hi),
               ('GBA+ / BL low',   gba_pos & ~bl_hi),
               ('GBA+ / BL high',  gba_pos &  bl_hi)]:
    if m.sum() < 5: continue
    curve = S_cum[m].mean(axis=0)
    ax.plot(t_axis, curve, label=f'{lab} (n={m.sum()})', marker='o')
ax.set_xlabel('months from baseline'); ax.set_ylabel('S(t) (not reached UPDRS≥40)')
ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.set_title('Fig 11 — NHH survival curves')
save_fig(fig, 'fig11_survival')

# Fig 12 — training curves
hist_ckpt = CHECKPOINTS/'block_06_full.pt'
if hist_ckpt.exists():
    ck = torch.load(hist_ckpt, map_location='cpu', weights_only=False)
    hist = ck.get('metadata', {}).get('history', [])
    if hist:
        fig, ax = plt.subplots(figsize=(7, 4))
        eps = [h['epoch']  for h in hist]
        ls  = [h['loss']   for h in hist]
        rs  = [h['r2_val'] for h in hist]
        ax.plot(eps, ls, label='train loss', color='steelblue')
        ax2 = ax.twinx(); ax2.plot(eps, rs, label='val R²', color='firebrick')
        ax.axvline(x=CFG['ughem_warmup'], color='k', linestyle=':', alpha=0.5, label='UG-HEM on')
        ax.axvline(x=CFG['kl_anneal_epochs'], color='g', linestyle=':', alpha=0.5, label='KL anneal end')
        ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax2.set_ylabel('val R²')
        ax.legend(loc='upper left'); ax2.legend(loc='upper right')
        ax.set_title('Fig 12 — training curves')
        save_fig(fig, 'fig12_training_curves')

# Fig 13 (NEW) — edge uncertainty atlas (GBA+, GBA-, Prodromal-NC)
pro_te = cohort_test == 'Prodromal'
event_te = test_out['event_ind'].astype(bool)
prodromal_nonconv = pro_te & ~event_te
groups = dict(
    GBA_plus = gba_pos,
    GBA_minus= ~gba_pos,
    Prodromal_nonconverter = prodromal_nonconv,
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (lbl, m) in zip(axes, groups.items()):
    if m.sum() < 5:
        ax.set_title(f'{lbl}: n={m.sum()} (insufficient)'); continue
    sns.heatmap(test_out['u_edge'][m].mean(axis=0), xticklabels=NODE_NAMES, yticklabels=NODE_NAMES,
                ax=ax, cmap='magma')
    ax.set_title(f'{lbl}  n={m.sum()}')
fig.suptitle('Fig 13 — edge uncertainty atlas (S-PBGL posterior)')
save_fig(fig, 'fig13_edge_uncertainty_atlas')

# Fig 14 (NEW) — SAA graph edge strips (SNc, LC)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].boxplot([snc_row[~saa_pos], snc_row[saa_pos]], tick_labels=['SAA-', 'SAA+'])
axes[0].set_title('A_i at SNc by SAA'); axes[0].set_ylabel('mean edge weight')
axes[1].boxplot([lc_row[~saa_pos], lc_row[saa_pos]], tick_labels=['SAA-', 'SAA+'])
axes[1].set_title('A_i at LC by SAA')
save_fig(fig, 'fig14_saa_edges')

# Fig 15 (NEW) — topology-phenotype mean adjacencies
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for cid in range(3):
    info = cluster_info[f'cluster_{cid}']
    ax = axes[cid]
    sns.heatmap(np.array(info['mean_A']), xticklabels=NODE_NAMES, yticklabels=NODE_NAMES,
                ax=ax, cmap='Blues')
    ax.set_title(f"cluster {cid}: {info['label']}  n={info['n']}")
fig.suptitle('Fig 15 — topology phenotype mean adjacencies')
save_fig(fig, 'fig15_topology_phenotypes')

# Fig 16 (NEW) — u_graph vs u_pred coloured by GBA
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(u_graph[~gba_pos], u_pred[~gba_pos], s=10, color='#888', alpha=0.5, label='GBA-')
ax.scatter(u_graph[ gba_pos], u_pred[ gba_pos], s=18, color='crimson', alpha=0.8, label='GBA+')
ax.set_xlabel('u_graph (mean posterior edge var.)'); ax.set_ylabel('u_pred (NIG epistemic)')
ax.legend(); ax.set_title(f'Fig 16 — two-source uncertainty  r={r_ug:.2f} p={p_ug:.2e}')
save_fig(fig, 'fig16_two_source_uncertainty')

print('\nAll figures rendered to', FIGURES)


## Block 17 — Final Summary

Consolidates every JSON under `RESULTS/` and prints the primary Table 1 + publication-gate decision (DaTscan r ≥ 0.50, ΔR² ≥ 0.05, silhouette > 0.30).

In [ ]:

# ==== Block 17 — final summary ====
summary = {}
for p in sorted(RESULTS.glob('*.json')):
    try:
        summary[p.stem] = json.loads(p.read_text())
    except Exception as e:
        summary[p.stem] = f'<failed: {e}>'

with open(RESULTS/'main_results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

def _get(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d: return default
        d = d[k]
    return d

bf_r2_pd  = _get(summary, 'block_07_metrics', 'UPDRS_PD', 'R2')
bf_r2_all = _get(summary, 'block_07_metrics', 'UPDRS_all', 'R2')
dat_r     = _get(summary, 'block_07_metrics', 'DaTscan_Pearson_r')
dat_p     = _get(summary, 'block_07_metrics', 'DaTscan_Pearson_p')
f1_w      = _get(summary, 'block_07_metrics', 'NSD_ISS_weighted_F1')
ece       = _get(summary, 'block_08_calibration', 'ECE')
unc_ratio = _get(summary, 'block_08_calibration', 'uncertainty_ratio')
abl_d     = _get(summary, 'block_09_ablation', default={})
b0 = _get(abl_d, 'B0', 'R2_PD'); b5 = _get(abl_d, 'B5', 'R2_PD')
d_r2 = (b5 - b0) if (b0 is not None and b5 is not None) else None
silh  = _get(summary, 'block_15_subtyping', 'silhouette')

print('='*60)
print('BrainFormer-PD v2.1 — Final Results')
print('='*60)
print(f'UPDRS-III R² (PD cohort, primary):  {bf_r2_pd}')
print(f'UPDRS-III R² (all test):            {bf_r2_all}')
print(f'ΔR² vs XGBoost+LTP (B5 − B0):        {d_r2}')
print(f'NSD-ISS Weighted F1:                 {f1_w}')
print(f'DaTscan Pearson r (p):               {dat_r} ({dat_p})')
print(f'ECE:                                 {ece}')
print(f'Uncertainty ratio (Q5/Q1):           {unc_ratio}')
print(f'Latent-space silhouette (k=4):       {silh}')

# Publication gate
def ok(v, op, thr):
    if v is None or (isinstance(v, float) and np.isnan(v)): return False
    return (v >= thr) if op == '>=' else (v > thr)

gate = dict(
    DaTscan_r_geq_0_50  = ok(dat_r, '>=', 0.50),
    delta_R2_geq_0_05   = ok(d_r2, '>=', 0.05),
    silhouette_gt_0_30  = ok(silh, '>', 0.30),
)
print('\nPublication gate:')
for k, v in gate.items(): print(f'  {k}: {"PASS" if v else "FAIL"}')
print('\nFull result dump saved to:', RESULTS/'main_results.json')
